# Step 2. L2 Logistic Regression, kNN, and XGBoost with TF-IDF and E5

Train three score classifiers: L2-regularized logistic classification, kNN,
and XGBoost. Compare **TF-IDF** and frozen **E5 with four extra features
(388 dimensions)**. Each question has a score distribution for each of the
11 candidate LLMs over the fixed values **0, 0.25, 0.5, 0.75 and 1**.

Read each distribution in two ways:

- **Hard class:** the score with the highest predicted probability.
- **Expected score:** the probability-weighted average of the five scores.

Both outputs share the same predicted dollar cost. The experiment uses train,
calibration and validation. Training-only cross-validation selects the logistic
regularization strength and the Ridge cost alpha. A separate kNN CV selects
score and cost neighbor counts using the same grouped training folds.
Calibration is the development split for comparing these predictors; no
probability calibrator is fitted here. Validation predictions are saved for
downstream routing-policy development. The test split is reserved for the
final complete system.

For a complete run, restart the kernel, keep `RUN_MODE="full"`, and choose
**Run All**. All predictors and their training-CV searches are recomputed from
the prepared inputs. Full results overwrite their filenames in the fixed `full/` output
directory; archived runs are preserved. Smoke checks use separate timestamped directories.

Sections 10–11 append LogNormal and Gamma cost predictors using the same
prepared inputs and grouped training folds. Section 12 compares all cost
predictors on calibration; their predictions can be paired with any of the
existing score distributions for later routing experiments.

Use `experiment_manifest.json` for the selected settings and `run_status.json`
to confirm that the output export completed.


## 1. Environment and paths

Use a Python environment with NumPy, pandas, SciPy, scikit-learn >= 1.6,
XGBoost >= 2.0, joblib, threadpoolctl, matplotlib, IPython and Jinja2. The prepared preprocessing objects
should be used with compatible package versions.

Keep the prepared files in this layout:

```text
project/
  data/processed/
    char_tfidf_50k/
    multilingual_e5_small_chunk480/
  2_Router_Training/
    TF-IDF_E5_Score_Classification_Router.ipynb
```

A layout with `1_data_preparation/data/processed/` is also supported. The code
searches the current working directory and its parents. If the data lives
elsewhere, set `DATA_PREPARATION_DIR` to the folder containing `data/`, or set
the `ERDOS_DATA_DIR` environment variable. No user-specific paths are required.
Copy the prepared data with the notebook, or run the project's data-preparation
steps before using this file.

`RUN_MODE="full"` uses all prepared rows. Restart the kernel and run every
cell in order. Results are written to
`experiment_outputs/score_classification_5class/full/`. A full rerun overwrites
the same generated filenames and preserves `archive_before_*/` folders.
The configuration checks ownership of existing results. Use the current-run
manifests to identify outputs; extra files from earlier settings are not
listed as new results. Prepared Step 1 inputs stay outside this directory.

`"smoke"` runs a small execution check in a separate timestamped folder and
does not touch `full/`. Its metrics are not a performance estimate. If a run
stops early, `run_status.json` remains `"running"`; only a finished export marks
it `"completed"`. Completion is recorded after the original predictors,
both additional cost experiments and the final comparisons have been saved.
A rerun can overwrite an owned incomplete run.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import inspect
import hashlib
import json
import os
import stat
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from scipy.sparse import load_npz
from scipy.stats import kurtosis, probplot, skew
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import GammaRegressor, LogisticRegression, Ridge
from sklearn.metrics import mean_gamma_deviance
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier, XGBRegressor
from threadpoolctl import threadpool_limits


In [2]:
RUN_MODE = "full"
DATA_PREPARATION_DIR = None  # Optional: folder containing data/.
OUTPUT_BASE_OVERRIDE = None  # Optional: output folder outside the project.
SEED = 42
SPLITS = ("train", "calibration", "validation")
# Base-predictor diagnostics use calibration; validation feeds routing policies.
EVALUATION_SPLIT = "calibration"
SMOKE_LIMITS = {"train": 512, "calibration": 128, "validation": 256}

score_support = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
K_VALUES = [1, 2, 3, 5, 10, 20, 40, 60, 80, 100]
QUERY_BATCH_SIZE = 128

# Independent LLM fits run in separate processes; use 1 for serial execution.
FIT_N_JOBS = 2

LINEAR_C_GRID = np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])
LINEAR_MAX_ITER = 300
LINEAR_RETRY_MAX_ITER = (1000, 2000)
LINEAR_TOL = 1e-3

XGB_DEVICE = "cpu"  # Set to "cuda" with a supported GPU installation.
XGB_N_JOBS = 4
XGB_ROUNDS = 10 if RUN_MODE == "smoke" else 100

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if XGB_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("XGB_DEVICE must be 'cpu' or 'cuda'.")


def resolve_preparation_dir(override=None):
    """Find a prepared-data folder, preferring the closest project root."""
    requested = override or os.environ.get("ERDOS_DATA_DIR")
    if requested:
        candidates = [Path(requested).expanduser().resolve()]
    else:
        working_dir = Path.cwd().resolve()
        candidates = [
            candidate
            for parent in (working_dir, *working_dir.parents)
            for candidate in (parent, parent / "1_data_preparation")
        ]
    required = (
        Path("data/processed/char_tfidf_50k/preprocessing.joblib"),
        Path("data/processed/multilingual_e5_small_chunk480/embedding_metadata.json"),
    )
    for candidate in dict.fromkeys(candidates):
        if all((candidate / relative).is_file() for relative in required):
            return candidate.resolve()
    raise FileNotFoundError(
        "Prepared TF-IDF and E5 files were not found. Set DATA_PREPARATION_DIR "
        "or ERDOS_DATA_DIR to the folder containing data/processed/."
    )


def prepare_output_directory(base, mode, stamp):
    """Overwrite owned result filenames while preserving archived runs."""
    base = Path(base).expanduser().resolve()
    if base == Path(base.anchor):
        raise ValueError("Choose a project output folder below the filesystem root.")
    if mode not in {"full", "smoke"}:
        raise ValueError("Output mode must be 'full' or 'smoke'.")
    target = base / ("full" if mode == "full" else f"smoke_seed{SEED}_{stamp}")
    try:
        target_info = target.lstat()
    except FileNotFoundError:
        target_info = None
    reparse_flag = getattr(stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0)
    if target_info is not None and (
        target.is_symlink()
        or getattr(target_info, "st_file_attributes", 0) & reparse_flag
    ):
        raise ValueError(f"Output directory cannot be a symlink or junction: {target}")
    if target.resolve() != base / target.name or target.parent != base:
        raise ValueError(f"Output directory must stay directly inside its base: {target}")
    if target_info is not None and not stat.S_ISDIR(target_info.st_mode):
        raise ValueError(f"Output path is not a directory: {target}")

    workflow = "erdos_score_classification_router"
    if mode == "full" and target_info is not None:
        active_entries = []
        for child in target.iterdir():
            info = child.lstat()
            if child.is_symlink() or getattr(info, "st_file_attributes", 0) & reparse_flag:
                raise ValueError(f"Output entries cannot be links or junctions: {child}")
            if child.name.startswith("archive_before_"):
                archive_marker = child / "archive_manifest.json"
                try:
                    archived = json.loads(archive_marker.read_text(encoding="utf-8"))
                except (OSError, ValueError) as error:
                    raise ValueError(f"Unrecognized archive folder: {child}") from error
                if archived.get("workflow") != "erdos_score_classification_router_archive":
                    raise ValueError(f"Unrecognized archive ownership: {child}")
            else:
                active_entries.append(child)
        if active_entries:
            try:
                previous = json.loads((target / "run_status.json").read_text(encoding="utf-8"))
            except (OSError, ValueError) as error:
                raise ValueError(f"Unrecognized existing output folder: {target}") from error
            if previous.get("workflow") != workflow or previous.get("mode") != "full":
                raise ValueError(f"Unrelated existing output folder: {target}")
        print("Full results will overwrite their saved filenames; archives are preserved:", target)

    target.mkdir(parents=True, exist_ok=(mode == "full"))
    status = {
        "workflow": workflow, "mode": mode, "status": "running", "run_id": stamp,
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "output_policy": ("overwrite_owned_files_preserve_archives"
                          if mode == "full" else "new_smoke_directory"),
    }
    (target / "run_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    return target


preparation_dir = resolve_preparation_dir(DATA_PREPARATION_DIR)
table_dir = preparation_dir / "data/processed/char_tfidf_50k"
e5_dir = preparation_dir / "data/processed/multilingual_e5_small_chunk480"
project_dir = preparation_dir.parent if preparation_dir.name == "1_data_preparation" else preparation_dir
output_base = (
    Path(OUTPUT_BASE_OVERRIDE).expanduser().resolve()
    if OUTPUT_BASE_OVERRIDE is not None
    else project_dir / "2_Router_Training/experiment_outputs/score_classification_5class"
)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = prepare_output_directory(output_base, RUN_MODE, run_stamp)
# Reset provenance on every run, including when an existing kernel is reused.
execution_provenance = {
    "refit_scope": "all_predictors",
    "prepared_inputs": "Step 1 TF-IDF and frozen E5 features",
    "output_policy": "overwrite_owned_files_preserve_archives" if RUN_MODE == "full" else "new_smoke_directory",
}
print("Mode:", RUN_MODE)
print("Prepared data:", preparation_dir)
print("Outputs:", run_dir)


Full results will overwrite their saved filenames; archives are preserved: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full
Mode: full
Prepared data: C:\Users\johnh\MachineLearning\project_erdos
Outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full


## 2. Load the prepared inputs

Reuse the question, score and dollar-cost tables, pure TF-IDF features, and
388-dimensional E5 features. Row indices and LLM column order must agree.
The original score tables stay unchanged throughout this notebook.


In [3]:
preprocessing = joblib.load(table_dir / "preprocessing.joblib")
tfidf_vectorizer_50k = preprocessing["tfidf_vectorizer"]
model_names = list(preprocessing["model_names"])
assert len(model_names) == 11
n_tfidf_features = len(tfidf_vectorizer_50k.vocabulary_)
e5_length_scaler = joblib.load(e5_dir / "length_scaler.joblib")
e5_count_scaler = joblib.load(e5_dir / "count_scaler.joblib")
feature_labels = {"tfidf": "TF-IDF", "e5_388": "E5 + features (388)"}
full_tables, full_features = {}, {feature: {} for feature in feature_labels}
for split in SPLITS:
    full_tables[split] = {name: pd.read_pickle(table_dir / f"{split}_{name}.pkl")
                         for name in ("questions", "scores", "costs")}
    q, s, c = (full_tables[split][name] for name in ("questions", "scores", "costs"))
    assert q.index.is_unique and q.index.equals(s.index) and q.index.equals(c.index)
    assert s.columns.tolist() == c.columns.tolist() == model_names
    assert np.isfinite(s.to_numpy(dtype=float)).all() and s.ge(0).all().all() and s.le(1).all().all()
    assert np.isfinite(c.to_numpy(dtype=float)).all() and c.gt(0).all().all()
    full_features["tfidf"][split] = load_npz(table_dir / f"X_{split}.npz")[:, :n_tfidf_features].astype(np.float32).tocsr()
    full_features["e5_388"][split] = np.load(e5_dir / f"X_{split}_e5_full_features.npy", allow_pickle=False)
    assert full_features["tfidf"][split].shape == (len(q), n_tfidf_features)
    assert full_features["e5_388"][split].shape == (len(q), 388)

# Keep the full tables for label coverage; only the execution subset changes.
tables, split_features = {}, {feature: {} for feature in feature_labels}
for split in SPLITS:
    q = full_tables[split]["questions"]
    positions = np.arange(len(q))
    if RUN_MODE == "smoke":
        rng = np.random.default_rng(SEED + SPLITS.index(split))
        positions = rng.choice(len(q), min(SMOKE_LIMITS[split], len(q)), replace=False)
        if split == "train":
            # Include each observed score to exercise the fixed mapping in smoke mode.
            scores = full_tables[split]["scores"].to_numpy(dtype=float)
            examples = [np.argwhere(scores == value)[0, 0] for value in np.unique(scores)]
            positions = np.unique(np.r_[positions, examples])
        positions.sort()
    tables[split] = {name: frame.iloc[positions].copy() for name, frame in full_tables[split].items()}
    for feature in feature_labels:
        split_features[feature][split] = full_features[feature][split][positions].copy()
    tables[split]["questions"][["sample_id"]].to_csv(run_dir / f"{split}_rows.csv")
train_questions, train_scores, train_costs = (tables["train"][name] for name in ("questions", "scores", "costs"))
evaluation_scores, evaluation_costs = (tables[EVALUATION_SPLIT][name] for name in ("scores", "costs"))
train_scores_arr = train_scores.to_numpy(dtype=float)  # Original recorded scores.
cost_means = train_costs.to_numpy(dtype=float).mean(axis=0)
train_cost_normalized = (train_costs.to_numpy(dtype=float) / cost_means).astype(np.float32)
train_groups = train_questions["question_group"].to_numpy()
assert len(train_questions) >= max(K_VALUES)
all_questions = pd.concat([tables[split]["questions"].assign(_split=split) for split in SPLITS])
assert all_questions["sample_id"].is_unique
for group_key in ("question_group", "question_text"):
    assert all_questions.groupby(group_key)["_split"].nunique().eq(1).all()
display(pd.DataFrame({"Questions": {split: len(tables[split]["questions"]) for split in SPLITS}}))
print("Fixed score classes:", score_support.tolist())


,Questions
train,21900
calibration,3648
validation,3649


Fixed score classes: [0.0, 0.25, 0.5, 0.75, 1.0]


## 3. Why five score classes?

The five anchors cover **99.858% of training scores**. The table below counts
individual question-LLM scores in the prepared files, before any mapping or
smoke subsetting. The other values are sparse and provide little support for
learning separate classes.

| Recorded score | Train count | Calibration count |
|---|---:|---:|
| 0 | 93,268 | 15,532 |
| 0.25 | 15,064 | 2,417 |
| 0.5 | 8,075 | 1,304 |
| 0.75 | 27,679 | 4,599 |
| 1 | 96,473 | 16,203 |
| Other values | 341 | 73 |
| **Five-anchor coverage** | **99.858%** | **99.818%** |

Using five classes gives a compact common output schema and reduces the number
of sparsely supported categories. It is a practical simplification; it does not
guarantee higher predictive accuracy or resolve uncertainty in the grading.
The next cell recomputes this table from the files loaded on your computer.

For classification, map each recorded score $s$ to its nearest fixed anchor:

$$z(s)=\arg\min_{j\in\{0,\ldots,4\}}|s-a_j|,
\qquad a=(0,\,0.25,\,0.5,\,0.75,\,1).$$

An exact midpoint uses the lower anchor. Current non-anchor values map as follows:

| Recorded values | Classification target |
|---|---:|
| 0.1 | 0 |
| 0.2, 0.3 | 0.25 |
| 0.4, 0.6 | 0.5 |
| 0.7, 0.8 | 0.75 |
| 0.9 | 1 |

The anchors and rule are fixed before evaluation. All questions remain in the
experiment. Classification accuracy uses mapped targets. Score MSE and MAE use
the **original recorded scores**, so their meaning is preserved.


In [4]:
coverage_rows = []
for label in [*score_support, "Other values"]:
    row = {"Recorded score": label}
    for split in ("train", EVALUATION_SPLIT):
        values = full_tables[split]["scores"].to_numpy(dtype=float)
        row[split] = int(
            (~np.isin(values, score_support)).sum()
            if isinstance(label, str) else (values == label).sum()
        )
    coverage_rows.append(row)
label_coverage_counts = pd.DataFrame(coverage_rows).set_index("Recorded score")
anchor_coverage = pd.DataFrame({
    split: {
        "Total score entries": full_tables[split]["scores"].size,
        "Five-anchor coverage": np.isin(
            full_tables[split]["scores"].to_numpy(dtype=float), score_support,
        ).mean(),
    }
    for split in ("train", EVALUATION_SPLIT)
}).T
display(label_coverage_counts)
display(anchor_coverage.style.format({
    "Total score entries": "{:,.0f}", "Five-anchor coverage": "{:.3%}",
}))
label_coverage_counts.to_csv(run_dir / "raw_score_value_counts.csv")
anchor_coverage.to_csv(run_dir / "raw_score_anchor_coverage.csv")


,train,calibration
Recorded score,,
0.0,93268,15532
0.25,15064,2417
0.5,8075,1304
0.75,27679,4599
1.0,96473,16203
Other values,341,73


,Total score entries,Five-anchor coverage
train,"240,900",99.858%
calibration,"40,128",99.818%


In [5]:
def encode_score_classes(values):
    """Map original scores to fixed anchors without changing the source array."""
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError("Scores must be finite and lie in [0, 1].")
    distances = np.abs(values[..., None] - score_support)
    return distances.argmin(axis=-1).astype(np.int64)


def align_class_probabilities(probabilities, classes, n_classes):
    probabilities = np.asarray(probabilities, dtype=float)
    classes = np.asarray(classes, dtype=int)
    aligned = np.zeros((len(probabilities), n_classes), dtype=float)
    aligned[:, classes] = probabilities
    if not np.isfinite(aligned).all() or np.any(aligned < 0) or not np.allclose(aligned.sum(axis=1), 1, atol=1e-6):
        raise ValueError("Invalid score probability distribution.")
    return aligned


def class_log_loss(targets, probabilities, floor=1e-12):
    true_probability = probabilities[np.arange(len(targets)), targets]
    return float(-np.log(np.maximum(true_probability, floor)).mean())


def prediction_frame(values, split):
    return pd.DataFrame(values, index=tables[split]["questions"].index, columns=model_names)


score_class_ids = encode_score_classes(train_scores_arr)
training_folds = list(GroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(
    train_scores_arr, groups=train_groups,
))
for fit_indices, heldout_indices in training_folds:
    assert not np.intersect1d(train_groups[fit_indices], train_groups[heldout_indices]).size
e5_train = split_features["e5_388"]["train"]
raw_lengths = e5_length_scaler.inverse_transform(e5_train[:, 384:385])
raw_counts = e5_count_scaler.inverse_transform(e5_train[:, 385:387])
training_texts = train_questions["question_text"].to_numpy()


def build_fold_features(feature, fit_indices, heldout_indices):
    if feature == "tfidf":
        vectorizer = clone(tfidf_vectorizer_50k)
        return (
            vectorizer.fit_transform(training_texts[fit_indices]).astype(np.float32).tocsr(),
            vectorizer.transform(training_texts[heldout_indices]).astype(np.float32).tocsr(),
        )
    length_scaler = clone(e5_length_scaler).fit(raw_lengths[fit_indices])
    count_scaler = clone(e5_count_scaler).fit(raw_counts[fit_indices])

    def transform(positions):
        return np.column_stack([e5_train[positions, :384],
                                length_scaler.transform(raw_lengths[positions]),
                                count_scaler.transform(raw_counts[positions]),
                                e5_train[positions, 387:388]]).astype(np.float32)

    return transform(fit_indices), transform(heldout_indices)


prediction_bundles = {}


## 4. L2-regularized logistic classification

Train one classifier per candidate LLM. For one LLM, let $N$ be the number of
training questions, $z_i$ the mapped score-class index of question $i$, and
$p_{i,z_i}$ its predicted probability for that class. Let $W$ contain the
class-weight vectors as rows; $\|W\|_F^2$ is the sum of their squared entries.
With regularization coefficient $\lambda$, the training objective is

$$\mathcal L_{\mathrm{score}}=
-\frac1N\sum_{i=1}^N\log p_{i,z_i}
+\frac\lambda2\|W\|_F^2.$$

Intercepts are unpenalized. Scikit-learn uses $C>0$, with
$\lambda=1/(NC)$ in this mean-loss formulation. Smaller C gives stronger
regularization.

Select one `C` per
feature set by mean cross-entropy over three training folds and the 11 LLMs.
Test C = 0.01, 0.1, 1, 10, 100, 1000 and 10000. This is a broad,
logarithmically spaced search. NLL selects C; Accuracy and the numerical
score errors below are auxiliary diagnostics.
Question groups stay within one fold. TF-IDF and the extra-feature scalers are
refitted inside each fold; the frozen E5 vectors are reused.

For held-out question $i$ and LLM $m$, let $p_{imj}$ be the predicted probability
of class $j$, let $v_j\in\{0,0.25,0.5,0.75,1\}$ be its score, and let $s_{im}$
be the original recorded score. The two scalar predictions are

$$\hat s^{\mathrm{expected}}_{im}=\sum_{j=0}^{4}p_{imj}v_j,
\qquad \hat s^{\mathrm{hard}}_{im}=v_{\arg\max_j p_{imj}}.$$

For either output $a$, the held-out numerical diagnostics are

$$\mathrm{MSE}^{(a)}=\frac1{n_H\,11}\sum_{i\in H}\sum_{m=1}^{11}
(\hat s^{(a)}_{im}-s_{im})^2,\qquad
\mathrm{MAE}^{(a)}=\frac1{n_H\,11}\sum_{i\in H}\sum_{m=1}^{11}
|\hat s^{(a)}_{im}-s_{im}|.$$

Here $H$ is a held-out fold and $n_H$ its number of questions. Fold metrics are
weighted by held-out question counts. Both outputs come from the same class
probabilities and share the C selected by the freshly computed CV NLL. These
errors use raw recorded scores; classification accuracy and NLL use the mapped
five-class labels.

The price head is a multi-output Ridge model. For LLM $m$, let
$\mu_m=\operatorname{mean}_{\mathrm{train}}c_{im}$. Fit normalized costs
$c_{im}/\mu_m$ with a squared-error objective and L2 penalty. Clip negative
estimates to a small positive value and multiply by $\mu_m$ to restore dollars.

The score-fit cell trains the selected Logistic classifiers. The Ridge
cost-fit cell then trains the CV-selected cost heads and assembles their shared
probability/cost bundles. A full run fits both parts from the prepared data.
Final Logistic classifiers must converge before evaluation and export continue.


In [6]:
L2_OPTIONS = {"penalty": "l2"} if inspect.signature(LogisticRegression).parameters["l1_ratio"].default is None else {"l1_ratio": 0.0}


def fit_linear_score_head(features, targets, C):
    """Return a classifier, or its constant class, with fit diagnostics."""
    classes = np.unique(targets)
    if classes.size == 1:
        return {
            "classifier": None, "constant_class": int(classes[0]),
            "iterations": 0, "converged": True, "observed_classes": classes,
            "convergence_warning_count": 0,
        }
    classifier = LogisticRegression(
        C=float(C), solver="lbfgs", max_iter=LINEAR_MAX_ITER, **L2_OPTIONS,
        tol=LINEAR_TOL, random_state=SEED,
    )
    total_iterations, warning_count = 0, 0
    for iteration_limit in (LINEAR_MAX_ITER, *LINEAR_RETRY_MAX_ITER):
        classifier.set_params(max_iter=iteration_limit)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            classifier.fit(features, targets)
        convergence_messages = [str(item.message) for item in caught
                                if issubclass(item.category, ConvergenceWarning)]
        warning_count += len(convergence_messages)
        total_iterations += int(classifier.n_iter_.max())
        for item in caught:
            if not issubclass(item.category, ConvergenceWarning):
                warnings.warn(str(item.message), item.category)
        if not convergence_messages:
            break
        # Continue from the current coefficients with the same objective and tolerance.
        classifier.set_params(warm_start=True)
        print(f"    iteration limit {iteration_limit}: continuing optimization", flush=True)
    if not np.isfinite(classifier.coef_).all() or not np.isfinite(classifier.intercept_).all():
        raise ValueError("A linear score classifier produced nonfinite parameters.")
    return {
        "classifier": classifier, "constant_class": None,
        "iterations": total_iterations,
        "converged": not convergence_messages,
        "observed_classes": classifier.classes_.astype(int),
        "convergence_warning_count": warning_count,
    }


def linear_score_probabilities(head, features):
    if head["classifier"] is None:
        probabilities = np.zeros((features.shape[0], len(score_support)))
        probabilities[:, head["constant_class"]] = 1.0
        return probabilities
    classifier = head["classifier"]
    return align_class_probabilities(
        classifier.predict_proba(features), classifier.classes_,
        len(score_support),
    )


def fit_linear_score_job(features, targets, C):
    """Fit one independent LLM score head and record its own training time."""
    started = perf_counter()
    head = fit_linear_score_head(features, targets, C)
    return head, perf_counter() - started


In [7]:
linear_cv_rows = []
linear_cv_head_rows = []
for feature in feature_labels:
    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        print(f"Linear score CV / {feature_labels[feature]} / fold {fold_id}: features", flush=True)
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        for C in LINEAR_C_GRID:
            started = perf_counter()
            losses, accuracies = [], []
            expected_mse, expected_mae, hard_mse, hard_mae = [], [], [], []
            unseen_labels = 0
            n_unconverged = 0
            warning_count = 0
            max_iterations = 0
            max_observed_classes = 0
            fitted_heads = joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes",
                           max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
                joblib.delayed(fit_linear_score_job)(
                    X_fit, score_class_ids[fit_indices, position], C,
                ) for position in range(len(model_names))
            )
            for model_position, model_name in enumerate(model_names):
                head, fit_seconds = fitted_heads[model_position]
                probabilities = linear_score_probabilities(head, X_heldout)
                targets = score_class_ids[heldout_indices, model_position]
                losses.append(class_log_loss(targets, probabilities))
                accuracies.append(float((probabilities.argmax(axis=1) == targets).mean()))
                raw_scores = train_scores_arr[heldout_indices, model_position]
                expected_scores = probabilities @ score_support
                hard_scores = score_support[probabilities.argmax(axis=1)]
                expected_errors = expected_scores - raw_scores
                hard_errors = hard_scores - raw_scores
                expected_mse.append(float(np.mean(expected_errors ** 2)))
                expected_mae.append(float(np.mean(np.abs(expected_errors))))
                hard_mse.append(float(np.mean(hard_errors ** 2)))
                hard_mae.append(float(np.mean(np.abs(hard_errors))))
                linear_cv_head_rows.append({
                    "Feature": feature, "Fold": fold_id, "C": float(C),
                    "Model": model_name, "Questions": len(heldout_indices),
                    "NLL": losses[-1], "Accuracy": accuracies[-1],
                    "Expected_Score_MSE": expected_mse[-1], "Expected_Score_MAE": expected_mae[-1],
                    "Hard_Score_MSE": hard_mse[-1], "Hard_Score_MAE": hard_mae[-1],
                    "Iterations": head["iterations"], "Converged": head["converged"],
                    "Convergence_warnings": head["convergence_warning_count"],
                    "Seconds": fit_seconds,
                })
                unseen_labels += int((~np.isin(targets, head["observed_classes"])).sum())
                n_unconverged += int(not head["converged"])
                warning_count += head["convergence_warning_count"]
                max_iterations = max(max_iterations, head["iterations"])
                max_observed_classes = max(max_observed_classes, len(head["observed_classes"]))
                print(
                    f"  C={C:g} / {model_position + 1}/{len(model_names)} / "
                    f"{model_name}: CE={losses[-1]:.4f}, "
                    f"iterations={head['iterations']}, converged={head['converged']}",
                    flush=True,
                )
            linear_cv_rows.append({
                "Feature": feature, "Fold": fold_id, "C": float(C),
                "Questions": len(heldout_indices), "Mean_CrossEntropy": np.mean(losses),
                "Mean_Accuracy": np.mean(accuracies),
                "Mean_Expected_Score_MSE": np.mean(expected_mse),
                "Mean_Expected_Score_MAE": np.mean(expected_mae),
                "Mean_Hard_Score_MSE": np.mean(hard_mse),
                "Mean_Hard_Score_MAE": np.mean(hard_mae),
                "Unseen_label_events": unseen_labels,
                "Unconverged_heads": n_unconverged, "Max_iterations": max_iterations,
                "Convergence_warnings": warning_count, "Max_observed_classes": max_observed_classes,
                "Seconds": perf_counter() - started,
            })
            # Persist completed candidate/fold diagnostics during the long search.
            pd.DataFrame(linear_cv_rows).to_csv(run_dir / "linear_cv_folds.csv", index=False)
            pd.DataFrame(linear_cv_head_rows).to_csv(run_dir / "linear_cv_heads.csv", index=False)
        del X_fit, X_heldout

linear_cv_table = pd.DataFrame(linear_cv_rows)
linear_cv_summary_rows = []
for (feature, C), rows in linear_cv_table.groupby(["Feature", "C"], sort=False):
    linear_cv_summary_rows.append({
        "Feature": feature, "C": C,
        "CV_CrossEntropy": np.average(rows["Mean_CrossEntropy"], weights=rows["Questions"]),
        "CV_Accuracy": np.average(rows["Mean_Accuracy"], weights=rows["Questions"]),
        "CV_Expected_Score_MSE": np.average(rows["Mean_Expected_Score_MSE"], weights=rows["Questions"]),
        "CV_Expected_Score_MAE": np.average(rows["Mean_Expected_Score_MAE"], weights=rows["Questions"]),
        "CV_Hard_Score_MSE": np.average(rows["Mean_Hard_Score_MSE"], weights=rows["Questions"]),
        "CV_Hard_Score_MAE": np.average(rows["Mean_Hard_Score_MAE"], weights=rows["Questions"]),
        "Unseen_label_events": int(rows["Unseen_label_events"].sum()),
        "Unconverged_heads": int(rows["Unconverged_heads"].sum()),
        "Convergence_warnings": int(rows["Convergence_warnings"].sum()),
        "Max_iterations": int(rows["Max_iterations"].max()),
        "Max_observed_classes": int(rows["Max_observed_classes"].max()),
        "Seconds": rows["Seconds"].sum(),
    })
linear_cv_summary = pd.DataFrame(linear_cv_summary_rows)
linear_cv_head_table = pd.DataFrame(linear_cv_head_rows)
linear_cv_head_table.to_csv(run_dir / "linear_cv_heads.csv", index=False)
linear_cv_summary.to_csv(run_dir / "linear_cv_summary.csv", index=False)
if linear_cv_summary["Unconverged_heads"].sum():
    raise RuntimeError("Resolve unconverged CV fits before selecting C; see linear_cv_heads.csv.")
selected_linear_C = {}
for feature in feature_labels:
    choices = linear_cv_summary.loc[linear_cv_summary["Feature"] == feature]
    # Exact ties favor the smaller C, which gives stronger regularization.
    best = choices.sort_values(["CV_CrossEntropy", "C"], kind="stable").iloc[0]
    selected_linear_C[feature] = float(best["C"])
display(linear_cv_summary)
print("Selected score-classifier C:", selected_linear_C)


Linear score CV / TF-IDF / fold 1: features


  C=0.01 / 1/11 / WizardLM/WizardLM-13B-V1.2: CE=0.8760, iterations=12, converged=True


  C=0.01 / 2/11 / claude-instant-v1: CE=0.8181, iterations=13, converged=True


  C=0.01 / 3/11 / claude-v1: CE=0.7928, iterations=15, converged=True


  C=0.01 / 4/11 / claude-v2: CE=0.8040, iterations=15, converged=True


...

  C=10000 / 9/11 / mistralai/mistral-7b-chat: CE=0.6650, iterations=99, converged=True


  C=10000 / 10/11 / mistralai/mixtral-8x7b-chat: CE=0.7295, iterations=125, converged=True


  C=10000 / 11/11 / zero-one-ai/Yi-34B-Chat: CE=0.6880, iterations=152, converged=True


,Feature,C,CV_CrossEntropy,CV_Accuracy,CV_Expected_Score_MSE,CV_Expected_Score_MAE,CV_Hard_Score_MSE,CV_Hard_Score_MAE,Unseen_label_events,Unconverged_heads,Convergence_warnings,Max_iterations,Max_observed_classes,Seconds
0,tfidf,0.01,0.790557,0.662989,0.166573,0.361716,0.261273,0.282146,0,0,0,17,5,59.873141
1,tfidf,0.10,0.672715,0.679232,0.156574,0.341097,0.245621,0.266277,0,0,0,31,5,81.788573
2,tfidf,1.00,0.644155,0.681826,0.157320,0.326248,0.246230,0.265879,0,0,0,71,5,145.517980
3,tfidf,10.00,0.740185,0.658016,0.181195,0.317665,0.267224,0.287534,0,0,0,120,5,251.767980
4,tfidf,100.00,1.059246,0.637742,0.215459,0.314919,0.281367,0.303009,0,0,0,139,5,324.071137
5,tfidf,1000.00,1.460150,0.634736,0.231332,0.313697,0.282645,0.304638,0,0,0,143,5,349.948062
6,tfidf,10000.00,1.535908,0.632603,0.233211,0.313880,0.283722,0.306012,0,0,0,138,5,327.543546
7,e5_388,0.01,0.705808,0.659128,0.166813,0.360067,0.262019,0.283542,0,0,0,28,5,8.100977
8,e5_388,0.10,0.666148,0.676359,0.158656,0.344548,0.246763,0.267672,0,0,0,47,5,12.322913
9,e5_388,1.00,0.641868,0.684027,0.156222,0.332207,0.243549,0.263335,0,0,0,91,5,22.802415


Selected score-classifier C: {'tfidf': 1.0, 'e5_388': 1.0}


### Cross-validation for Ridge alpha

Select one Ridge alpha for each representation using the same three grouped
training folds. Candidate values are **0.01, 0.1, 1, 10, 100, 1000 and 10000**.

The selection metric is **mean Cost WAPE across the 11 LLMs**. Let $N$ be the
number of training questions, $m=1,\ldots,11$ index LLMs, and $c_{im}$ be the
recorded dollar cost for training question $i$. For candidate penalty $\alpha$,
let $\hat c_{im}^{(\alpha)}$ be the dollar prediction made while question $i$ is
held out. Every training question receives one such out-of-fold prediction.
Define the cost-selection criterion as

$$
\mathrm{CV\_CostWAPE}(\alpha)=\frac1{11}\sum_{m=1}^{11}
\frac{\sum_{i=1}^{N}|\hat c_{im}^{(\alpha)}-c_{im}|}
{\sum_{i=1}^{N}c_{im}}.
$$

Lower is better. Aggregate dollar errors over all held-out questions before
forming each LLM's ratio. The individual fold WAPEs are diagnostics. Exact ties
favor larger alpha. A best value at a grid boundary calls for a wider search.

Within each fold, refit TF-IDF or the E5 auxiliary-feature scalers on the fitting
rows. Normalize each LLM's fitting costs by its **fold-fitting mean**, predict
normalized costs, apply the positive floor, and restore dollars using that same
mean. Calibration, validation and test labels do not participate in selection.

Save the CV tables, row IDs, out-of-fold predictions and selection settings.
The Ridge cost-fit cell then trains the final heads using `selected_cost_ridge_alpha`.


In [8]:
# Select Ridge alpha with grouped cross-validation on training data.
COST_RIDGE_ALPHA_GRID = np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])
cost_cv_targets = train_costs.loc[:, model_names].to_numpy(dtype=np.float64)
cost_cv_n_questions, cost_cv_n_models = cost_cv_targets.shape
assert cost_cv_n_models == 11
assert train_costs.index.equals(train_questions.index)
assert np.isfinite(cost_cv_targets).all() and (cost_cv_targets > 0).all()
assert len(training_folds) == 3

# Every training question must be held out once, with no group leakage.
cost_cv_holdout_counts = np.zeros(cost_cv_n_questions, dtype=int)
cost_cv_fold_ids = np.zeros(cost_cv_n_questions, dtype=int)
for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
    assert not np.intersect1d(fit_indices, heldout_indices).size
    assert len(fit_indices) + len(heldout_indices) == cost_cv_n_questions
    assert not np.intersect1d(train_groups[fit_indices], train_groups[heldout_indices]).size
    cost_cv_holdout_counts[heldout_indices] += 1
    cost_cv_fold_ids[heldout_indices] = fold_id
assert np.all(cost_cv_holdout_counts == 1)

if RUN_MODE == "smoke":
    warnings.warn("Cost CV is using smoke rows. Use full mode before interpreting alpha choices.")

cost_cv_output_dir = run_dir / "cost_ridge_cv"
cost_cv_output_dir.mkdir(parents=True, exist_ok=True)
cost_cv_fold_rows, cost_cv_model_rows, cost_cv_summary_rows = [], [], []
cost_cv_oof_predictions = {}
cost_cv_started = perf_counter()

for feature in feature_labels:
    # Store dollar predictions in original training-row order for each alpha.
    predictions = np.full(
        (len(COST_RIDGE_ALPHA_GRID), cost_cv_n_questions, cost_cv_n_models),
        np.nan, dtype=np.float64,
    )
    normalized_sse = np.zeros(len(COST_RIDGE_ALPHA_GRID), dtype=np.float64)
    fit_seconds = np.zeros(len(COST_RIDGE_ALPHA_GRID), dtype=np.float64)

    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        print(f"Ridge CV / {feature_labels[feature]} / fold {fold_id}: prepare features", flush=True)
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        fold_cost_means = cost_cv_targets[fit_indices].mean(axis=0)
        fit_targets = (cost_cv_targets[fit_indices] / fold_cost_means).astype(np.float32)
        heldout_costs = cost_cv_targets[heldout_indices]
        assert X_fit.shape[0] == len(fit_indices)
        assert X_heldout.shape[0] == len(heldout_indices)

        for alpha_position, alpha in enumerate(COST_RIDGE_ALPHA_GRID):
            started = perf_counter()
            cost_regressor = Ridge(alpha=float(alpha), solver="lsqr", tol=1e-3, max_iter=1000)
            cost_regressor.fit(X_fit, fit_targets)
            predicted_costs = np.maximum(cost_regressor.predict(X_heldout), 1e-6) * fold_cost_means
            assert predicted_costs.shape == heldout_costs.shape
            assert np.isfinite(predicted_costs).all() and (predicted_costs > 0).all()
            predictions[alpha_position, heldout_indices] = predicted_costs

            errors = predicted_costs - heldout_costs
            normalized_sse[alpha_position] += np.square(errors / fold_cost_means).sum()
            elapsed = perf_counter() - started
            fit_seconds[alpha_position] += elapsed
            fold_wape = np.mean(np.abs(errors).sum(axis=0) / heldout_costs.sum(axis=0))
            iterations = int(np.max(cost_regressor.n_iter_))
            cost_cv_fold_rows.append({
                "Feature": feature, "Fold": fold_id, "alpha": float(alpha),
                "Fit_questions": len(fit_indices), "Heldout_questions": len(heldout_indices),
                "Fold_Cost_WAPE": float(fold_wape),
                "Fold_Cost_MSE_normalized": float(np.square(errors / fold_cost_means).mean()),
                "Max_iterations": iterations, "Seconds": elapsed,
            })
            # Fold WAPEs are diagnostic; selection uses all OOF rows below.
            print(f"  alpha={alpha:g}: fold WAPE={fold_wape:.3%}, iterations={iterations}, seconds={elapsed:.1f}", flush=True)
            pd.DataFrame(cost_cv_fold_rows).to_csv(cost_cv_output_dir / "cost_cv_folds.csv", index=False)
            if iterations >= 1000:
                warnings.warn(f"Cost Ridge reached the iteration limit: {feature}, fold {fold_id}, alpha {alpha:g}.")

        del X_fit, X_heldout, cost_regressor

    assert np.isfinite(predictions).all()
    cost_cv_oof_predictions[feature] = predictions
    for alpha_position, alpha in enumerate(COST_RIDGE_ALPHA_GRID):
        errors = predictions[alpha_position] - cost_cv_targets
        # Sum dollar errors across every held-out row BEFORE forming WAPE ratios.
        model_wapes = np.abs(errors).sum(axis=0) / cost_cv_targets.sum(axis=0)
        cost_cv_summary_rows.append({
            "Feature": feature, "alpha": float(alpha),
            "CV_Cost_WAPE": float(model_wapes.mean()),
            "CV_Cost_MAE": float(np.abs(errors).mean()),
            "CV_Cost_MSE": float(np.square(errors).mean()),
            "CV_Cost_MSE_normalized": float(normalized_sse[alpha_position] / cost_cv_targets.size),
            "Fit_seconds": float(fit_seconds[alpha_position]),
        })
        for model_position, model_name in enumerate(model_names):
            cost_cv_model_rows.append({
                "Feature": feature, "alpha": float(alpha), "Model": model_name,
                "CV_Cost_WAPE": float(model_wapes[model_position]),
                "CV_Cost_MAE": float(np.abs(errors[:, model_position]).mean()),
                "CV_Cost_MSE": float(np.square(errors[:, model_position]).mean()),
            })

cost_cv_summary = pd.DataFrame(cost_cv_summary_rows)
selected_cost_ridge_alpha = {}
for feature in feature_labels:
    choices = cost_cv_summary.loc[cost_cv_summary["Feature"].eq(feature)]
    # Exact ties favor larger alpha.
    best = choices.sort_values(["CV_Cost_WAPE", "alpha"], ascending=[True, False], kind="stable").iloc[0]
    selected_cost_ridge_alpha[feature] = float(best["alpha"])
    if best["alpha"] in (COST_RIDGE_ALPHA_GRID.min(), COST_RIDGE_ALPHA_GRID.max()):
        warnings.warn(f"Best cost alpha is on the grid boundary for {feature}; consider extending the grid.")

cost_cv_summary["Selected"] = [
    alpha == selected_cost_ridge_alpha[feature]
    for feature, alpha in cost_cv_summary[["Feature", "alpha"]].itertuples(index=False, name=None)
]
cost_cv_summary.to_csv(cost_cv_output_dir / "cost_cv_summary.csv", index=False)
pd.DataFrame(cost_cv_model_rows).to_csv(cost_cv_output_dir / "cost_cv_per_llm.csv", index=False)
cost_cv_rows = train_questions.loc[:, ["sample_id", "question_group"]].copy()
cost_cv_rows["Fold"] = cost_cv_fold_ids
cost_cv_rows.to_csv(cost_cv_output_dir / "cost_cv_training_rows.csv")
for feature, predictions in cost_cv_oof_predictions.items():
    np.savez_compressed(cost_cv_output_dir / f"cost_cv_oof_{feature}.npz",
                        alpha=COST_RIDGE_ALPHA_GRID, dollar_cost=predictions)

cost_cv_manifest = {
    "experiment": "Ridge alpha selection", "mode": RUN_MODE, "seed": SEED,
    "training_questions": cost_cv_n_questions, "model_names": model_names,
    "n_folds": len(training_folds), "alpha_grid": COST_RIDGE_ALPHA_GRID.tolist(),
    "selection_metric": "mean across 11 LLMs of per-LLM WAPE over all out-of-fold training rows",
    "normalization": "LLM cost mean from each fold's fitting rows only",
    "feature_preprocessing": "TF-IDF and auxiliary scalers refitted within each fold",
    "tie_rule": "larger alpha on exact WAPE ties",
    "selected_alpha": selected_cost_ridge_alpha,
    "calibration_used_for_selection": False, "validation_used_for_selection": False,
    "test_used": False,
    "oof_array_shape": [len(COST_RIDGE_ALPHA_GRID), cost_cv_n_questions, cost_cv_n_models],
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "elapsed_seconds": perf_counter() - cost_cv_started,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(cost_cv_output_dir / "cost_cv_manifest.json").write_text(json.dumps(cost_cv_manifest, indent=2), encoding="utf-8")

display(cost_cv_summary.style.format({
    "alpha": "{:g}", "CV_Cost_WAPE": "{:.3%}", "CV_Cost_MAE": "${:.8f}",
    "CV_Cost_MSE": "{:.3e}", "CV_Cost_MSE_normalized": "{:.6f}", "Fit_seconds": "{:.1f}",
}))
print("Selected Ridge alpha:", selected_cost_ridge_alpha)
print("Cost CV outputs:", cost_cv_output_dir)
print("Ridge alpha selection complete; the next cells fit the final score and cost predictors.")


Ridge CV / TF-IDF / fold 1: prepare features


  alpha=0.01: fold WAPE=28.503%, iterations=97, seconds=31.9


  alpha=0.1: fold WAPE=21.963%, iterations=59, seconds=20.7


  alpha=1: fold WAPE=17.786%, iterations=28, seconds=9.5


  alpha=10: fold WAPE=17.243%, iterations=13, seconds=5.0


...

  alpha=100: fold WAPE=13.828%, iterations=6, seconds=0.2


  alpha=1000: fold WAPE=15.380%, iterations=5, seconds=0.2


  alpha=10000: fold WAPE=23.791%, iterations=4, seconds=0.1


,Feature,alpha,CV_Cost_WAPE,CV_Cost_MAE,CV_Cost_MSE,CV_Cost_MSE_normalized,Fit_seconds,Selected
0,tfidf,0.01,28.655%,$0.00026508,5.119e-07,0.195960,88.5,False
1,tfidf,0.1,22.026%,$0.00021296,3.979e-07,0.140587,60.8,False
2,tfidf,1,17.710%,$0.00017437,3.565e-07,0.121314,29.1,False
3,tfidf,10,17.143%,$0.00016572,3.840e-07,0.128012,14.1,True
4,tfidf,100,21.130%,$0.00019832,4.885e-07,0.159566,7.7,False
5,tfidf,1000,37.441%,$0.00034136,8.347e-07,0.286352,4.4,False
6,tfidf,10000,56.701%,$0.00053619,1.566e-06,0.545980,3.3,False
7,e5_388,0.01,14.858%,$0.00016266,3.117e-07,0.102912,1.7,False
8,e5_388,0.1,14.770%,$0.00016149,3.118e-07,0.102915,1.6,False
9,e5_388,1,14.210%,$0.00015438,3.148e-07,0.103156,1.5,False


Selected Ridge alpha: {'tfidf': 10.0, 'e5_388': 10.0}
Cost CV outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full\cost_ridge_cv
Ridge alpha selection complete; the next cells fit the final score and cost predictors.


In [9]:
# Fit the CV-selected score classifiers; Ridge costs are fitted in the next cell.
linear_models = {}
linear_score_predictions, linear_fit_rows = {}, []
for feature in feature_labels:
    C = selected_linear_C[feature]
    X_train = split_features[feature]["train"]
    fitted_heads = joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes",
                                 max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
        joblib.delayed(fit_linear_score_job)(X_train, score_class_ids[:, position], C)
        for position in range(len(model_names))
    )
    heads = []
    for model_position, model_name in enumerate(model_names):
        head, fit_seconds = fitted_heads[model_position]
        heads.append(head)
        linear_fit_rows.append({
            "Feature": feature, "Model": model_name, "C": C,
            "Iterations": head["iterations"], "Converged": head["converged"],
            "Convergence_warnings": head["convergence_warning_count"],
            "Observed_classes": len(head["observed_classes"]), "Seconds": fit_seconds,
        })
        print(f"Logistic / {feature_labels[feature]} / {model_position + 1}/{len(model_names)}: {model_name}")
    linear_models[feature] = {"heads": heads, "selected_C": C}
    for split in ("calibration", "validation"):
        X = split_features[feature][split]
        linear_score_predictions[(feature, split)] = np.stack(
            [linear_score_probabilities(head, X) for head in heads], axis=1,
        )

linear_fit_table = pd.DataFrame(linear_fit_rows)
linear_fit_table.to_csv(run_dir / "linear_final_fit.csv", index=False)
display(linear_fit_table)
if (~linear_fit_table["Converged"]).any():
    raise RuntimeError("Resolve unconverged final Logistic fits before evaluation/export; see linear_final_fit.csv.")


Logistic / TF-IDF / 1/11: WizardLM/WizardLM-13B-V1.2
Logistic / TF-IDF / 2/11: claude-instant-v1
Logistic / TF-IDF / 3/11: claude-v1
Logistic / TF-IDF / 4/11: claude-v2
Logistic / TF-IDF / 5/11: gpt-3.5-turbo-1106
Logistic / TF-IDF / 6/11: gpt-4-1106-preview
Logistic / TF-IDF / 7/11: meta/code-llama-instruct-34b-chat
Logistic / TF-IDF / 8/11: meta/llama-2-70b-chat
Logistic / TF-IDF / 9/11: mistralai/mistral-7b-chat
Logistic / TF-IDF / 10/11: mistralai/mixtral-8x7b-chat
Logistic / TF-IDF / 11/11: zero-one-ai/Yi-34B-Chat


Logistic / E5 + features (388) / 1/11: WizardLM/WizardLM-13B-V1.2
Logistic / E5 + features (388) / 2/11: claude-instant-v1
Logistic / E5 + features (388) / 3/11: claude-v1
Logistic / E5 + features (388) / 4/11: claude-v2
Logistic / E5 + features (388) / 5/11: gpt-3.5-turbo-1106
Logistic / E5 + features (388) / 6/11: gpt-4-1106-preview
Logistic / E5 + features (388) / 7/11: meta/code-llama-instruct-34b-chat
Logistic / E5 + features (388) / 8/11: meta/llama-2-70b-chat
Logistic / E5 + features (388) / 9/11: mistralai/mistral-7b-chat
Logistic / E5 + features (388) / 10/11: mistralai/mixtral-8x7b-chat
Logistic / E5 + features (388) / 11/11: zero-one-ai/Yi-34B-Chat


,Feature,Model,C,Iterations,Converged,Convergence_warnings,Observed_classes,Seconds
0,tfidf,WizardLM/WizardLM-13B-V1.2,1.0,30,True,0,5,7.971947
1,tfidf,claude-instant-v1,1.0,40,True,0,5,10.331901
2,tfidf,claude-v1,1.0,31,True,0,5,7.803039
3,tfidf,claude-v2,1.0,54,True,0,5,13.528224
4,tfidf,gpt-3.5-turbo-1106,1.0,55,True,0,5,15.250126
5,tfidf,gpt-4-1106-preview,1.0,45,True,0,5,13.944622
6,tfidf,meta/code-llama-instruct-34b-chat,1.0,22,True,0,5,5.677435
7,tfidf,meta/llama-2-70b-chat,1.0,66,True,0,5,17.310623
8,tfidf,mistralai/mistral-7b-chat,1.0,41,True,0,5,10.290150
9,tfidf,mistralai/mixtral-8x7b-chat,1.0,65,True,0,5,16.671427


In [10]:
# Fit Ridge cost heads independently of Logistic score selection.
for feature in feature_labels:
    cost_model = Ridge(
        alpha=selected_cost_ridge_alpha[feature], solver="lsqr", tol=1e-3, max_iter=1000,
    ).fit(split_features[feature]["train"], train_cost_normalized)
    linear_models[feature]["cost_model"] = cost_model
    for split in ("calibration", "validation"):
        costs = np.maximum(cost_model.predict(split_features[feature][split]), 1e-6) * cost_means[None, :]
        prediction_bundles[("L2 logistic", feature, split)] = {
            "probability": linear_score_predictions[(feature, split)],
            "cost": prediction_frame(costs, split),
        }
display(pd.DataFrame({
    "Feature": list(feature_labels),
    "Applied_cost_alpha": [linear_models[feature]["cost_model"].alpha for feature in feature_labels],
}))


,Feature,Applied_cost_alpha
0,tfidf,10.0
1,e5_388,10.0


Read score-classifier CV, Ridge CV and convergence tables separately.
Logistic C and Ridge alpha are selected independently using training data.
The next methods use their own fitting cells; a complete run retrains every
predictor before the shared calibration evaluation and final export.


## 5. kNN class frequencies and mean dollar cost

Let $x$ be a question's feature vector and $x_i$ a training vector. Cosine
distance retrieves similar training questions:

$$d(x,x_i)=1-\frac{x^\mathsf{T}x_i}{\|x\|\|x_i\|}.$$

For a candidate neighbor count $k$, let $N_k(x)$ be the indices of its $k$
nearest training questions. For LLM $m$ and score class $j$, $z(s_{im})$ is
training question $i$'s mapped class and $c_{im}$ its recorded dollar cost.
The uniform neighbor estimates are

$$p_{mj}^{(k)}(x)=\frac1k\sum_{i\in N_k(x)}\mathbf1[z(s_{im})=j],
\qquad
\hat c_m^{(k)}(x)=\frac1k\sum_{i\in N_k(x)}c_{im}.$$

Expected score averages the mapped neighbor scores; hard class selects the
most frequent mapped class. All LLMs use the same retrieved question indices.
Queries are processed in batches, retrieving the largest requested neighborhood
once and reusing its prefixes for every candidate $k$.

Each representation receives two neighbor counts: $k_{\mathrm{score}}$ for
class probabilities and $k_{\mathrm{cost}}$ for dollar costs. Training CV selects
them independently below. The main comparison combines probabilities from
$k_{\mathrm{score}}$ with costs from $k_{\mathrm{cost}}$. Hard and expected score
outputs share these probabilities and cost predictions.


In [11]:
def _checked_knn_k_values(reference_size):
    """Validate the shared neighbor grid against the fitted reference rows."""
    if not K_VALUES or any(int(value) != value or value < 1 for value in K_VALUES):
        raise ValueError("K_VALUES must contain positive integer neighbor counts.")
    values = sorted(int(value) for value in K_VALUES)
    if len(set(values)) != len(values):
        raise ValueError("K_VALUES must not contain duplicates.")
    if values[-1] > reference_size:
        raise ValueError(
            f"The largest k ({values[-1]}) exceeds the reference size ({reference_size})."
        )
    return values


def predict_neighbor_arrays(index, features, reference_class_ids, reference_costs,
                            query_name):
    """Predict all k values from explicitly supplied, index-aligned references.

    Returns float64 probability and dollar-cost arrays. Neighbor positions refer
    to these reference arrays, not to the full training table's row positions.
    """
    reference_class_ids = np.asarray(reference_class_ids)
    reference_costs = np.asarray(reference_costs, dtype=np.float64)
    n_models, n_classes = len(model_names), len(score_support)
    if reference_class_ids.ndim != 2 or reference_class_ids.shape[1] != n_models:
        raise ValueError("Reference classes must have shape (reference rows, LLMs).")
    if not np.issubdtype(reference_class_ids.dtype, np.integer):
        raise ValueError("Reference class IDs must be integers.")
    if np.any((reference_class_ids < 0) | (reference_class_ids >= n_classes)):
        raise ValueError("Reference class IDs fall outside the score support.")
    if reference_costs.shape != reference_class_ids.shape:
        raise ValueError("Reference costs and class IDs must have identical shapes.")
    if not np.isfinite(reference_costs).all() or np.any(reference_costs <= 0):
        raise ValueError("Reference dollar costs must be finite and positive.")
    n_reference = len(reference_class_ids)
    if getattr(index, "n_samples_fit_", n_reference) != n_reference:
        raise ValueError("The fitted index and reference outcomes have different row counts.")
    k_values = _checked_knn_k_values(n_reference)
    if int(QUERY_BATCH_SIZE) != QUERY_BATCH_SIZE or QUERY_BATCH_SIZE < 1:
        raise ValueError("QUERY_BATCH_SIZE must be a positive integer.")
    batch_size = int(QUERY_BATCH_SIZE)
    count = features.shape[0]
    predictions = {
        k: {
            "probability": np.empty((count, n_models, n_classes), dtype=np.float64),
            "cost": np.empty((count, n_models), dtype=np.float64),
        }
        for k in k_values
    }
    started = perf_counter()
    max_k = k_values[-1]
    for batch_number, start in enumerate(range(0, count, batch_size), 1):
        stop = min(start + batch_size, count)
        neighbors = index.kneighbors(
            features[start:stop], n_neighbors=max_k, return_distance=False,
        )
        if neighbors.shape != (stop - start, max_k):
            raise ValueError("Unexpected neighbor-array shape.")
        labels = reference_class_ids[neighbors]
        cost_sums = np.cumsum(reference_costs[neighbors], axis=1, dtype=np.float64)
        class_counts = np.zeros((stop - start, n_models, n_classes), dtype=np.float64)
        rows, models = np.indices((stop - start, n_models))
        for position in range(max_k):
            class_counts[rows, models, labels[:, position, :]] += 1.0
            k = position + 1
            if k in predictions:
                predictions[k]["probability"][start:stop] = class_counts / k
                predictions[k]["cost"][start:stop] = cost_sums[:, position, :] / k
        if batch_number == 1 or batch_number % 10 == 0 or stop == count:
            print(
                f"kNN / {query_name}: {stop}/{count} questions | "
                f"{perf_counter() - started:.1f}s", flush=True,
            )
    return predictions


def predict_neighbor_classes(index, features, split):
    """Keep the original split-based interface for final training-reference queries."""
    predictions = predict_neighbor_arrays(
        index, features, score_class_ids,
        train_costs.loc[:, model_names].to_numpy(dtype=np.float64), split,
    )
    for bundle in predictions.values():
        bundle["cost"] = prediction_frame(bundle["cost"], split)
    return predictions


### Select neighbor counts inside training

Test $k\in\{1,2,3,5,10,20,40,60,80,100\}$ using the same three grouped
training folds as the linear models. In each fold, refit TF-IDF or the E5
auxiliary-feature scalers on the fitting rows. Build a neighbor index from
those rows only, then predict the held-out training questions. A held-out
question and every member of its question group stay outside its neighbor
library. Each training question receives one out-of-fold (OOF) prediction.

Let $N$ be the number of training questions, $m=1,\ldots,11$ index LLMs,
$z_{im}$ be the mapped recorded class, and $\hat z_{im}^{(k)}$ the OOF hard-class
prediction. Score selection maximizes macro Accuracy:

$$\mathrm{CVAccuracy}(k)=\frac1{11}\sum_{m=1}^{11}
\frac1N\sum_{i=1}^{N}\mathbf1[\hat z_{im}^{(k)}=z_{im}].$$

Let $\hat c_{im}^{(k)}$ denote the OOF dollar-cost prediction. Cost selection
minimizes macro WAPE:

$$\mathrm{CVCostWAPE}(k)=\frac1{11}\sum_{m=1}^{11}
\frac{\sum_{i=1}^{N}|\hat c_{im}^{(k)}-c_{im}|}
{\sum_{i=1}^{N}c_{im}}.$$

Pool all OOF rows before forming each LLM's WAPE. Select $k_{\mathrm{score}}$
and $k_{\mathrm{cost}}$ separately for TF-IDF and E5. Exact ties favor the
larger $k$, giving a smoother estimate. NLL and score MSE/MAE are auxiliary
diagnostics; they do not select $k$. Calibration and validation labels are
excluded from this search. A selected value at the upper boundary should be
reported before extending the grid in a later experiment.


In [12]:
# Select score and cost neighbor counts independently using training rows only.
# Prerequisites: the shared fold/preprocessing helpers and kNN helper cell.
knn_cv_k_values = _checked_knn_k_values(len(train_questions))
knn_cv_scores = train_scores.loc[:, model_names].to_numpy(dtype=np.float64)
knn_cv_costs = train_costs.loc[:, model_names].to_numpy(dtype=np.float64)
knn_cv_classes = encode_score_classes(knn_cv_scores)
knn_cv_n_questions, knn_cv_n_models = knn_cv_scores.shape
knn_cv_n_classes = len(score_support)
knn_cv_probability_floor = 1e-12
assert knn_cv_n_models == 11 and knn_cv_n_classes == 5
assert train_scores.index.equals(train_costs.index)
assert train_questions.index.equals(train_scores.index)
assert np.array_equal(knn_cv_classes, score_class_ids)
assert np.isfinite(knn_cv_costs).all() and (knn_cv_costs > 0).all()
assert len(training_folds) == 3
knn_cv_groups = train_questions["question_group"].to_numpy()
knn_cv_holdout_counts = np.zeros(knn_cv_n_questions, dtype=np.int64)
knn_cv_fold_ids = np.zeros(knn_cv_n_questions, dtype=np.int64)
for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
    assert len(fit_indices) and len(heldout_indices)
    assert np.array_equal(
        np.sort(np.concatenate([fit_indices, heldout_indices])),
        np.arange(knn_cv_n_questions),
    )
    assert not np.intersect1d(fit_indices, heldout_indices).size
    assert not np.intersect1d(
        knn_cv_groups[fit_indices], knn_cv_groups[heldout_indices],
    ).size
    assert len(fit_indices) >= max(knn_cv_k_values)
    knn_cv_holdout_counts[heldout_indices] += 1
    knn_cv_fold_ids[heldout_indices] = fold_id
assert np.all(knn_cv_holdout_counts == 1)


def knn_training_oof_metrics(probabilities, predicted_costs, recorded_scores,
                             recorded_classes, recorded_costs):
    """Return per-LLM metrics; all supplied rows belong to the training split."""
    count, n_models = recorded_scores.shape
    assert probabilities.shape == (count, n_models, len(score_support))
    assert predicted_costs.shape == recorded_costs.shape == recorded_scores.shape
    assert recorded_classes.shape == recorded_scores.shape
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1.0, atol=1e-6)
    assert np.isfinite(predicted_costs).all() and (predicted_costs > 0).all()
    hard_classes = probabilities.argmax(axis=-1)
    true_probabilities = np.take_along_axis(
        probabilities, recorded_classes[..., None], axis=-1,
    )[..., 0]
    balanced_accuracy = []
    for model_position in range(n_models):
        actual = recorded_classes[:, model_position]
        predicted = hard_classes[:, model_position]
        recalls = [
            np.mean(predicted[actual == value] == value)
            for value in np.unique(actual)
        ]
        balanced_accuracy.append(float(np.mean(recalls)))
    expected_scores = probabilities @ score_support
    cost_errors = predicted_costs - recorded_costs
    return pd.DataFrame({
        "Accuracy": (hard_classes == recorded_classes).mean(axis=0),
        "Balanced_accuracy": balanced_accuracy,
        "NLL": -np.log(np.maximum(true_probabilities, knn_cv_probability_floor)).mean(axis=0),
        "Score_MSE": np.square(expected_scores - recorded_scores).mean(axis=0),
        "Score_MAE": np.abs(expected_scores - recorded_scores).mean(axis=0),
        "Cost_MSE": np.square(cost_errors).mean(axis=0),
        "Cost_MAE": np.abs(cost_errors).mean(axis=0),
        "Cost_WAPE": np.abs(cost_errors).sum(axis=0) / recorded_costs.sum(axis=0),
    }, index=pd.Index(model_names, name="Model"))


knn_cv_output_dir = run_dir / "knn_cv"
knn_cv_output_dir.mkdir(parents=True, exist_ok=True)
knn_cv_fold_rows, knn_cv_summary_rows, knn_cv_per_llm_frames = [], [], []
selected_knn_score_k, selected_knn_cost_k = {}, {}
knn_cv_selected_rows = []
knn_cv_started = perf_counter()
knn_cv_metric_names = [
    "Accuracy", "Balanced_accuracy", "NLL", "Score_MSE", "Score_MAE",
    "Cost_MSE", "Cost_MAE", "Cost_WAPE",
]

for feature in feature_labels:
    feature_started = perf_counter()
    # Pooled tensors preserve original training-row order, independent of folds.
    oof_probabilities = np.full(
        (len(knn_cv_k_values), knn_cv_n_questions, knn_cv_n_models, knn_cv_n_classes),
        np.nan, dtype=np.float64,
    )
    oof_costs = np.full(
        (len(knn_cv_k_values), knn_cv_n_questions, knn_cv_n_models),
        np.nan, dtype=np.float64,
    )
    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        started = perf_counter()
        print(
            f"kNN training CV / {feature_labels[feature]} / fold {fold_id}: "
            f"{len(fit_indices)} fitting, {len(heldout_indices)} held-out questions",
            flush=True,
        )
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        assert X_fit.shape[0] == len(fit_indices)
        assert X_heldout.shape[0] == len(heldout_indices)
        neighbor_index = NearestNeighbors(
            metric="cosine", algorithm="brute", n_jobs=-1,
        ).fit(X_fit)
        fold_predictions = predict_neighbor_arrays(
            neighbor_index, X_heldout, knn_cv_classes[fit_indices],
            knn_cv_costs[fit_indices], f"training CV {feature} fold {fold_id}",
        )
        fold_seconds = perf_counter() - started
        for k_position, k in enumerate(knn_cv_k_values):
            bundle = fold_predictions[k]
            oof_probabilities[k_position, heldout_indices] = bundle["probability"]
            oof_costs[k_position, heldout_indices] = bundle["cost"]
            per_llm = knn_training_oof_metrics(
                bundle["probability"], bundle["cost"], knn_cv_scores[heldout_indices],
                knn_cv_classes[heldout_indices], knn_cv_costs[heldout_indices],
            )
            means = per_llm.mean(axis=0)
            knn_cv_fold_rows.append({
                "Feature": feature, "Fold": fold_id, "k": k,
                "Fit_questions": len(fit_indices),
                "Heldout_questions": len(heldout_indices),
                **{f"Fold_{name}": float(means[name]) for name in knn_cv_metric_names},
                # All k values share this retrieval/preprocessing duration.
                "Shared_fold_seconds": fold_seconds,
            })
        pd.DataFrame(knn_cv_fold_rows).to_csv(
            knn_cv_output_dir / "knn_cv_folds.csv", index=False,
        )
        del X_fit, X_heldout, neighbor_index, fold_predictions, bundle, per_llm
    assert np.isfinite(oof_probabilities).all() and np.isfinite(oof_costs).all()
    feature_rows = []
    for k_position, k in enumerate(knn_cv_k_values):
        # WAPE ratios and class recalls are formed from ALL held-out rows here,
        # rather than averaging per-fold ratios or per-fold balanced accuracies.
        per_llm = knn_training_oof_metrics(
            oof_probabilities[k_position], oof_costs[k_position],
            knn_cv_scores, knn_cv_classes, knn_cv_costs,
        )
        means = per_llm.mean(axis=0)
        feature_rows.append({
            "Feature": feature, "k": k,
            **{f"CV_{name}": float(means[name]) for name in knn_cv_metric_names},
        })
        knn_cv_per_llm_frames.append(
            per_llm.rename(columns={name: f"CV_{name}" for name in knn_cv_metric_names})
            .reset_index().assign(Feature=feature, k=k)
        )
    feature_summary = pd.DataFrame(feature_rows)
    score_choice = feature_summary.sort_values(
        ["CV_Accuracy", "k"], ascending=[False, False], kind="stable",
    ).iloc[0]
    cost_choice = feature_summary.sort_values(
        ["CV_Cost_WAPE", "k"], ascending=[True, False], kind="stable",
    ).iloc[0]
    score_k, cost_k = int(score_choice["k"]), int(cost_choice["k"])
    selected_knn_score_k[feature], selected_knn_cost_k[feature] = score_k, cost_k
    if score_k == max(knn_cv_k_values) or cost_k == max(knn_cv_k_values):
        warnings.warn(
            f"A selected k reaches the upper grid boundary for {feature}; "
            "the current result is optimal only among the tested candidates."
        )
    for row in feature_rows:
        row["Selected_score"] = row["k"] == score_k
        row["Selected_cost"] = row["k"] == cost_k
    knn_cv_summary_rows.extend(feature_rows)
    knn_cv_selected_rows.append({
        "Feature": feature, "Selected_score_k": score_k, "Selected_cost_k": cost_k,
        "Selected_CV_Accuracy": float(score_choice["CV_Accuracy"]),
        "Selected_CV_Cost_WAPE": float(cost_choice["CV_Cost_WAPE"]),
        "Shared_feature_seconds": perf_counter() - feature_started,
    })
    np.savez_compressed(
        knn_cv_output_dir / f"knn_cv_oof_{feature}.npz",
        score_probability=oof_probabilities[knn_cv_k_values.index(score_k)],
        dollar_cost=oof_costs[knn_cv_k_values.index(cost_k)],
        score_k=np.asarray(score_k, dtype=np.int64),
        cost_k=np.asarray(cost_k, dtype=np.int64),
        fold_ids=knn_cv_fold_ids,
    )
    print(
        f"kNN training CV / {feature_labels[feature]}: "
        f"score k={score_k}, cost k={cost_k}", flush=True,
    )
    del oof_probabilities, oof_costs, per_llm

knn_cv_summary = pd.DataFrame(knn_cv_summary_rows)
knn_cv_folds = pd.DataFrame(knn_cv_fold_rows)
knn_cv_per_llm = pd.concat(knn_cv_per_llm_frames, ignore_index=True)
knn_cv_per_llm = knn_cv_per_llm[[
    "Feature", "k", "Model", *[f"CV_{name}" for name in knn_cv_metric_names],
]]
knn_cv_selection = pd.DataFrame(knn_cv_selected_rows)
assert len(knn_cv_summary) == len(feature_labels) * len(knn_cv_k_values)
assert len(knn_cv_folds) == len(feature_labels) * len(training_folds) * len(knn_cv_k_values)
assert len(knn_cv_per_llm) == len(knn_cv_summary) * knn_cv_n_models
knn_cv_summary.to_csv(knn_cv_output_dir / "knn_cv_summary.csv", index=False)
knn_cv_folds.to_csv(knn_cv_output_dir / "knn_cv_folds.csv", index=False)
knn_cv_per_llm.to_csv(knn_cv_output_dir / "knn_cv_per_llm.csv", index=False)
knn_cv_selection.to_csv(knn_cv_output_dir / "knn_cv_selection.csv", index=False)
knn_cv_training_rows = train_questions.loc[:, ["sample_id", "question_group"]].copy()
knn_cv_training_rows["Fold"] = knn_cv_fold_ids
knn_cv_training_rows.to_csv(knn_cv_output_dir / "knn_cv_training_rows.csv")

knn_cv_manifest = {
    "experiment": "training-only grouped cross-validation for kNN neighbor counts",
    "mode": RUN_MODE, "seed": SEED, "training_only": True,
    "calibration_used_for_selection": False,
    "validation_used_for_selection": False, "test_used": False,
    "training_questions": knn_cv_n_questions,
    "model_names": model_names, "score_support": np.asarray(score_support).tolist(),
    "k_values": knn_cv_k_values, "n_folds": len(training_folds),
    "folds": [{
        "Fold": fold_id, "Fit_questions": len(fit_indices),
        "Heldout_questions": len(heldout_indices),
    } for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1)],
    "group_column": "question_group", "each_training_row_held_out_once": True,
    "neighbor_index": "brute-force cosine; fitting rows only",
    "feature_preprocessing": "TF-IDF and auxiliary-feature scalers refitted on each fold's fitting rows",
    "score_selection": "highest equal-LLM mean pooled out-of-fold mapped-class Accuracy",
    "cost_selection": "lowest equal-LLM mean per-LLM Cost WAPE over all out-of-fold rows",
    "metric_aggregation": "pool out-of-fold rows before per-LLM metrics; then mean the 11 LLMs",
    "numerical_score_prediction": "expected mapped score; compared with original recorded score",
    "cost_prediction": "uniform mean of original fitting-neighbor dollar costs",
    "probability_floor": knn_cv_probability_floor,
    "tie_rule": "larger k on exact primary-metric ties",
    "selected_knn_score_k": selected_knn_score_k,
    "selected_knn_cost_k": selected_knn_cost_k,
    "saved_oof_arrays": "selected score probabilities and selected cost predictions only, with their independent k values and fold IDs",
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "elapsed_seconds": perf_counter() - knn_cv_started,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(knn_cv_output_dir / "knn_cv_manifest.json").write_text(
    json.dumps(knn_cv_manifest, indent=2), encoding="utf-8",
)
display(knn_cv_summary.style.format({
    "k": "{:d}", "CV_Accuracy": "{:.2%}", "CV_Balanced_accuracy": "{:.2%}",
    "CV_NLL": "{:.5f}", "CV_Score_MSE": "{:.5f}", "CV_Score_MAE": "{:.5f}",
    "CV_Cost_MSE": "{:.3e}", "CV_Cost_MAE": "${:.8f}", "CV_Cost_WAPE": "{:.3%}",
}))
display(knn_cv_selection)
print("Selected kNN score k:", selected_knn_score_k)
print("Selected kNN cost k:", selected_knn_cost_k)
print("Training-only kNN CV outputs:", knn_cv_output_dir)


kNN training CV / TF-IDF / fold 1: 14601 fitting, 7299 held-out questions


kNN / training CV tfidf fold 1: 128/7299 questions | 0.7s


kNN / training CV tfidf fold 1: 1280/7299 questions | 9.6s


kNN / training CV tfidf fold 1: 2560/7299 questions | 20.0s


kNN / training CV tfidf fold 1: 3840/7299 questions | 31.4s


...

kNN / training CV e5_388 fold 3: 7303/7303 questions | 4.8s


kNN training CV / E5 + features (388): score k=80, cost k=10


,Feature,k,CV_Accuracy,CV_Balanced_accuracy,CV_NLL,CV_Score_MSE,CV_Score_MAE,CV_Cost_MSE,CV_Cost_MAE,CV_Cost_WAPE,Selected_score,Selected_cost
0,tfidf,1,60.39%,43.39%,10.94413,0.29944,0.32470,7.012e-07,$0.00023105,25.632%,False,False
1,tfidf,2,58.41%,43.19%,6.27374,0.22727,0.32502,6.165e-07,$0.00021897,24.450%,False,False
2,tfidf,3,63.24%,43.54%,4.22483,0.20246,0.32503,5.879e-07,$0.00021469,24.032%,False,False
3,tfidf,5,64.62%,43.69%,2.54546,0.18272,0.32588,5.606e-07,$0.00021274,23.952%,False,True
4,tfidf,10,66.05%,43.92%,1.37034,0.16794,0.32670,5.457e-07,$0.00021454,24.252%,False,False
5,tfidf,20,67.00%,43.75%,0.91652,0.16129,0.32844,5.482e-07,$0.00022185,25.217%,False,False
6,tfidf,40,67.15%,43.44%,0.75998,0.15862,0.33034,5.531e-07,$0.00023246,26.711%,False,False
7,tfidf,60,67.21%,43.34%,0.72074,0.15796,0.33149,5.541e-07,$0.00023801,27.559%,False,False
8,tfidf,80,67.22%,43.20%,0.70611,0.15778,0.33230,5.559e-07,$0.00024191,28.138%,True,False
9,tfidf,100,67.20%,43.07%,0.69988,0.15777,0.33317,5.575e-07,$0.00024504,28.586%,False,False


,Feature,Selected_score_k,Selected_cost_k,Selected_CV_Accuracy,Selected_CV_Cost_WAPE,Shared_feature_seconds
0,tfidf,80,5,0.672154,0.239518,228.878504
1,e5_388,80,10,0.679195,0.103515,15.388510


Selected kNN score k: {'tfidf': 80, 'e5_388': 80}
Selected kNN cost k: {'tfidf': 5, 'e5_388': 10}
Training-only kNN CV outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full\knn_cv


In [13]:
knn_indices, knn_class_predictions = {}, {}
for feature in feature_labels:
    score_k = selected_knn_score_k[feature]
    cost_k = selected_knn_cost_k[feature]
    index = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(
        split_features[feature]["train"],
    )
    knn_indices[feature] = index
    knn_class_predictions[feature] = {}
    for split in ("calibration", "validation"):
        all_k = predict_neighbor_classes(index, split_features[feature][split], split)
        knn_class_predictions[feature][split] = all_k
        prediction_bundles[("kNN", feature, split)] = {
            "probability": all_k[score_k]["probability"],
            "cost": all_k[cost_k]["cost"],
        }
    print(f"Selected kNN / {feature_labels[feature]}: score k={score_k}, cost k={cost_k}")


kNN / calibration: 128/3648 questions | 1.1s


kNN / calibration: 1280/3648 questions | 14.1s


kNN / calibration: 2560/3648 questions | 28.6s


kNN / calibration: 3648/3648 questions | 39.7s


kNN / validation: 128/3649 questions | 1.1s


...

kNN / validation: 128/3649 questions | 0.2s


kNN / validation: 1280/3649 questions | 1.7s


kNN / validation: 2560/3649 questions | 3.2s


kNN / validation: 3649/3649 questions | 4.2s


Selected kNN / E5 + features (388): score k=80, cost k=10


## 6. XGBoost score classification and price prediction

Fit one boosted-tree score classifier and one price model per LLM. The score
head uses multiclass cross-entropy (`multi:softprob`). If a training subset has
only two observed classes, it uses binary cross-entropy. A constant target
produces a constant distribution. In every case, align the output to the same
five class columns and fill unobserved classes with zero probability.

The price head minimizes squared error on normalized costs. Restore dollar
units with the training cost means. Both hard and expected score outputs share
these price predictions. Tree settings are fixed in this experiment; XGBoost
hyperparameters are not selected by cross-validation here.


In [14]:
xgb_params = {
    "n_estimators": XGB_ROUNDS, "max_depth": 3, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8, "max_bin": 64,
    "tree_method": "hist", "device": XGB_DEVICE, "n_jobs": XGB_N_JOBS,
    "reg_lambda": 1.0, "random_state": SEED,
}


def fit_xgb_score_head(features, targets):
    classes, encoded = np.unique(targets, return_inverse=True)
    if len(classes) == 1:
        return {"classifier": None, "classes": classes}
    objective = "binary:logistic" if len(classes) == 2 else "multi:softprob"
    options = {"eval_metric": "logloss" if len(classes) == 2 else "mlogloss"}
    if len(classes) > 2:
        options["num_class"] = len(classes)
    classifier = XGBClassifier(objective=objective, **options, **xgb_params).fit(features, encoded)
    actual_device = json.loads(classifier.get_booster().save_config())["learner"]["generic_param"]["device"]
    if XGB_DEVICE == "cuda" and not actual_device.startswith("cuda"):
        raise RuntimeError("XGBoost fell back to CPU. Set XGB_DEVICE='cpu' or check the GPU installation.")
    return {"classifier": classifier, "classes": classes}


def xgb_score_probabilities(head, features):
    if head["classifier"] is None:
        probabilities = np.zeros((features.shape[0], len(score_support)))
        probabilities[:, head["classes"][0]] = 1.0
        return probabilities
    return align_class_probabilities(head["classifier"].predict_proba(features), head["classes"], len(score_support))


def fit_xgb_pair(features, score_targets, cost_targets):
    """Fit the independent score/cost pair for one candidate LLM."""
    started = perf_counter()
    head = fit_xgb_score_head(features, score_targets)
    price_head = XGBRegressor(objective="reg:squarederror", **xgb_params).fit(
        features, cost_targets,
    )
    return head, price_head, perf_counter() - started


In [15]:
xgb_models = {}
for feature in feature_labels:
    X_train = split_features[feature]["train"]
    xgb_models[feature] = {"score_classifier": [], "cost_regressor": []}
    raw = {split: {"probability": [], "cost": []} for split in ("calibration", "validation")}
    print(f"XGBoost / {feature_labels[feature]}: fitting {len(model_names)} score/cost pairs", flush=True)
    fitted_pairs = joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes",
                           max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
        joblib.delayed(fit_xgb_pair)(
            X_train, score_class_ids[:, position], train_cost_normalized[:, position],
        ) for position in range(len(model_names))
    )
    for model_position, model_name in enumerate(model_names):
        print(f"XGBoost / {feature_labels[feature]} / {model_position + 1}/{len(model_names)}: {model_name}", flush=True)
        head, price_head, fit_seconds = fitted_pairs[model_position]
        xgb_models[feature]["score_classifier"].append(head)
        xgb_models[feature]["cost_regressor"].append(price_head)
        for split in raw:
            X = split_features[feature][split]
            raw[split]["probability"].append(xgb_score_probabilities(head, X))
            costs = np.maximum(price_head.predict(X), 1e-6) * cost_means[model_position]
            raw[split]["cost"].append(costs)
        print(f"  completed in {fit_seconds:.1f}s", flush=True)
    for split, values in raw.items():
        prediction_bundles[("XGBoost", feature, split)] = {
            "probability": np.stack(values["probability"], axis=1),
            "cost": prediction_frame(np.column_stack(values["cost"]), split),
        }


XGBoost / TF-IDF: fitting 11 score/cost pairs


XGBoost / TF-IDF / 1/11: WizardLM/WizardLM-13B-V1.2


  completed in 346.7s


XGBoost / TF-IDF / 2/11: claude-instant-v1


  completed in 338.0s


...

  completed in 19.1s


XGBoost / E5 + features (388) / 10/11: mistralai/mixtral-8x7b-chat


  completed in 19.6s


XGBoost / E5 + features (388) / 11/11: zero-one-ai/Yi-34B-Chat


  completed in 13.1s


## 7. Evaluate probabilities and the two score outputs

**Evaluation split:** all predictor comparisons here use calibration.
Training-only grouped CV selects Logistic C, Ridge alpha, and separate kNN
score/cost neighbor counts. Use calibration to compare the selected predictors,
input representations and hard/expected score outputs. Validation predictions are exported for downstream routing-policy
development; this section does not evaluate validation outcomes.

The displayed tables use the freshly fitted models and their calibration
predictions. Preserve the untouched test
split for evaluating the fixed complete system.

For question $i$ and candidate LLM $m$:

$$\hat s^{\mathrm{hard}}_{im}=a_{\arg\max_j p_{imj}},
\qquad \hat s^{\mathrm{expected}}_{im}=\sum_{j=0}^4 a_jp_{imj}.$$

Hard class belongs to the five anchors. Expected score can take an intermediate
value, such as 0.63. An exact probability tie uses the lower anchor because the
class columns are ordered from 0 to 1.

**Probability metrics** evaluate the five-class target $z_{im}$:

$$\mathrm{Accuracy}_m=\frac1n\sum_i
\mathbf1[\arg\max_j p_{imj}=z_{im}],
\qquad
\mathrm{NLL}_m=-\frac1n\sum_i\log\max(p_{im,z_{im}},10^{-12}).$$

Balanced accuracy averages recall across the classes present in calibration.
It reveals performance hidden by frequent classes. Report the fraction of
zero true-class probabilities alongside NLL; the numerical floor keeps NLL
finite. Accuracy measures hard class decisions, while NLL evaluates the whole
distribution.

**Numerical error metrics** compare both score outputs with the original recorded
score $s_{im}$. Predicted costs are compared with recorded dollar costs $c_{im}$.
For MSE and MAE below, $y_{im}$ denotes either the original score or the
recorded cost, and $\hat y_{im}$ is the corresponding prediction:

$$\mathrm{MSE}_m=\frac1n\sum_i(\hat y_{im}-y_{im})^2,
\qquad \mathrm{MAE}_m=\frac1n\sum_i|\hat y_{im}-y_{im}|,$$

**Cost WAPE** is used only for dollar-cost prediction:

$$\mathrm{CostWAPE}_m=
\frac{\sum_i|\hat c_{im}-c_{im}|}{\sum_i c_{im}}.$$

Compute each metric separately for each LLM, then average equally across the
11 LLMs. Score prediction is evaluated with MSE and MAE on its original 0-to-1
scale. Cost prediction uses MSE, MAE and WAPE. All recorded costs are positive.
Hard and expected outputs share price predictions, so their cost errors are
identical. Probability metrics above remain unchanged.


In [16]:
NLL_EPSILON = 1e-12
score_support = np.asarray(score_support, dtype=float)
assert np.all(np.diff(score_support) > 0), "The score support must be strictly sorted."
evaluation_index = evaluation_scores.index
assert evaluation_costs.index.equals(evaluation_index), "Calibration score/cost rows must align."
evaluation_truth = evaluation_scores.loc[:, model_names].to_numpy(dtype=float)
evaluation_cost_truth = evaluation_costs.loc[:, model_names].to_numpy(dtype=float)
n_questions, n_models = evaluation_truth.shape
n_score_classes = len(score_support)


evaluation_class_ids = encode_score_classes(evaluation_truth)


def checked_probability_bundle(bundle):
    probabilities = np.asarray(bundle["probability"], dtype=float)
    costs = bundle["cost"]
    if probabilities.shape != (n_questions, n_models, n_score_classes):
        raise ValueError("Probabilities must have shape (questions, candidate LLMs, classes).")
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0):
        raise ValueError("Class probabilities must be finite and nonnegative.")
    if not np.allclose(probabilities.sum(axis=-1), 1.0, atol=1e-6):
        raise ValueError("Each candidate LLM needs a probability distribution summing to one.")
    if not costs.index.equals(evaluation_index) or costs.columns.tolist() != model_names:
        raise ValueError("Cost prediction rows and model columns must preserve calibration order.")
    cost_values = costs.to_numpy(dtype=float)
    if not np.isfinite(cost_values).all() or np.any(cost_values < 0):
        raise ValueError("Predicted dollar costs must be finite and nonnegative.")
    return probabilities, cost_values


def distribution_metrics(bundle):
    probabilities, _ = checked_probability_bundle(bundle)
    hard_class_ids = probabilities.argmax(axis=-1)
    true_probabilities = np.take_along_axis(
        probabilities, evaluation_class_ids[..., None], axis=-1,
    )[..., 0]
    balanced_accuracy = []
    for model_position in range(n_models):
        truth = evaluation_class_ids[:, model_position]
        prediction = hard_class_ids[:, model_position]
        recalls = [(prediction[truth == value] == value).mean() for value in np.unique(truth)]
        balanced_accuracy.append(float(np.mean(recalls)))
    return pd.DataFrame({
        "Accuracy": (hard_class_ids == evaluation_class_ids).mean(axis=0),
        "Balanced_accuracy": balanced_accuracy,
        "NLL": -np.log(np.maximum(true_probabilities, NLL_EPSILON)).mean(axis=0),
        "Zero_true_probability_rate": (true_probabilities == 0).mean(axis=0),
    }, index=pd.Index(model_names, name="model"))


def score_outputs(bundle):
    probabilities, costs = checked_probability_bundle(bundle)
    expected_scores = probabilities @ score_support
    return {
        "Expected score": {"score": expected_scores, "cost": costs},
        "Hard class": {"score": score_support[probabilities.argmax(axis=-1)], "cost": costs},
    }


def cost_wape(predicted_costs, recorded_costs):
    """Return one aggregate relative dollar-cost error for each LLM."""
    totals = np.abs(recorded_costs).sum(axis=0)
    return np.divide(
        np.abs(predicted_costs - recorded_costs).sum(axis=0), totals,
        out=np.full(recorded_costs.shape[1], np.nan), where=totals > 0,
    )


def numeric_prediction_metrics(bundle):
    scores, costs = bundle["score"], bundle["cost"]
    return pd.DataFrame({
        "Score_MSE": np.square(scores - evaluation_truth).mean(axis=0),
        "Score_MAE": np.abs(scores - evaluation_truth).mean(axis=0),
        "Cost_MSE": np.square(costs - evaluation_cost_truth).mean(axis=0),
        "Cost_MAE": np.abs(costs - evaluation_cost_truth).mean(axis=0),
        "Cost_WAPE": cost_wape(costs, evaluation_cost_truth),
    }, index=pd.Index(model_names, name="model"))


def training_frequency_bundle(train_scores, train_costs):
    train_values = train_scores.loc[:, model_names].to_numpy(dtype=float)
    train_classes = encode_score_classes(train_values)
    frequencies = np.stack([
        np.bincount(train_classes[:, model], minlength=n_score_classes) / len(train_values)
        for model in range(n_models)
    ])
    mean_costs = train_costs.loc[:, model_names].mean(axis=0).to_numpy(dtype=float)
    return {
        "probability": np.broadcast_to(frequencies, (n_questions, n_models, n_score_classes)),
        "cost": pd.DataFrame(np.broadcast_to(mean_costs, (n_questions, n_models)),
                             index=evaluation_index, columns=model_names),
    }


In [17]:
# prediction_bundles is keyed by (method, embedding, split).
evaluation_probability_bundles = {
    (method, embedding): bundle
    for (method, embedding, split), bundle in prediction_bundles.items()
    if split == EVALUATION_SPLIT
}
evaluation_probability_bundles = {
    ("Training frequency", "Baseline"): training_frequency_bundle(train_scores, train_costs),
    **evaluation_probability_bundles,
}
distribution_per_model = pd.concat({
    key: distribution_metrics(bundle)
    for key, bundle in evaluation_probability_bundles.items()
}, names=["method", "embedding"])
distribution_summary = distribution_per_model.groupby(
    level=["method", "embedding"], sort=False,
).mean()
numeric_evaluation_bundles = {
    (method, embedding, output): values
    for (method, embedding), bundle in evaluation_probability_bundles.items()
    for output, values in score_outputs(bundle).items()
}
numeric_per_model = pd.concat({
    key: numeric_prediction_metrics(bundle)
    for key, bundle in numeric_evaluation_bundles.items()
}, names=["method", "embedding", "score_output"])
numeric_summary = numeric_per_model.groupby(
    level=["method", "embedding", "score_output"], sort=False,
).mean()

print("Calibration probability quality: one result per trained classifier")
display(distribution_summary.style.format({
    "Accuracy": "{:.1%}", "Balanced_accuracy": "{:.1%}", "NLL": "{:.4f}",
    "Zero_true_probability_rate": "{:.1%}",
}))
print("Calibration score/cost errors: two score outputs from the same probabilities")
display(numeric_summary.style.format({
    "Score_MSE": "{:.5f}", "Score_MAE": "{:.5f}",
    "Cost_MSE": "{:.3e}", "Cost_MAE": "${:.8f}", "Cost_WAPE": "{:.1%}",
}, na_rep="undefined"))
distribution_summary.to_csv(run_dir / f"{EVALUATION_SPLIT}_classification_summary.csv")
distribution_per_model.to_csv(run_dir / f"{EVALUATION_SPLIT}_classification_per_llm.csv")
numeric_summary.to_csv(run_dir / f"{EVALUATION_SPLIT}_numeric_summary.csv")
numeric_per_model.to_csv(run_dir / f"{EVALUATION_SPLIT}_numeric_per_llm.csv")


Calibration probability quality: one result per trained classifier


Calibration score/cost errors: two score outputs from the same probabilities


## 8. Save and reuse the results

Each run saves probability arrays, hard scores, expected scores and dollar costs
for calibration and validation. DataFrames preserve question indices and LLM
columns. Probability arrays use `(questions, 11 LLMs, 5 classes)`; the manifest
records both column orders.

`predictions_manifest.csv` identifies the main six predictor/feature combinations.
Its kNN rows record `score_k` and `cost_k` separately. Use these prefixes for the
CV-selected main predictor. `knn_predictions_manifest.csv` additionally exports
every candidate k for both splits; each of those grid bundles uses the same k
for scores and cost. Its selection flags show which component was chosen.

The `knn_cv/` folder contains train-only fold, pooled OOF, per-LLM and selection
records. Calibration metric CSVs evaluate the selected predictors; validation
prediction files remain available for downstream routing-policy development.

The folder also contains class-coverage tables, linear-model CV records, model
settings and fitted Logistic/Ridge/XGBoost objects. Prediction on new questions
requires the same prepared TF-IDF/E5 workflow. Rebuild kNN indices from the
prepared training features, mapped score labels and recorded cost labels, using
the selected score and cost k values in the manifest.

For full runs, all files below belong to the fixed `full/` directory. Each
complete rerun overwrites the same result filenames and preserves archived
runs. Smoke outputs remain separate.
`run_status.json` records the actual run state and exported prediction count;
`experiment_manifest.json` records the selected parameters for that run.
The training-frequency baseline appears in calibration metric CSVs; it has no
separate prediction bundle. The prepared Step 1 files are kept outside this
output directory and are required again for new-text inference.

The appended LogNormal and Gamma experiments save **cost predictions only**,
with separate family manifests. Their files share this output directory; the
original probability/hard/expected/cost bundle layout stays the same. The global
run becomes completed only in the final comparison/export check.


In [18]:
def save_prediction_bundle(prefix, bundle, split):
    probabilities = np.asarray(bundle["probability"], dtype=float)
    assert probabilities.shape == (len(tables[split]["questions"]), len(model_names), 5)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1, atol=1e-6)
    np.save(run_dir / f"{prefix}_probability.npy", probabilities, allow_pickle=False)
    prediction_frame(probabilities @ score_support, split).to_pickle(run_dir / f"{prefix}_expected_score.pkl")
    prediction_frame(score_support[probabilities.argmax(axis=-1)], split).to_pickle(run_dir / f"{prefix}_hard_score.pkl")
    costs = bundle["cost"]
    assert costs.index.equals(tables[split]["questions"].index)
    assert costs.columns.tolist() == model_names
    assert np.isfinite(costs.to_numpy(dtype=float)).all() and costs.gt(0).all().all()
    costs.to_pickle(run_dir / f"{prefix}_dollar_cost.pkl")


prediction_manifest = []
for position, ((method, feature, split), bundle) in enumerate(prediction_bundles.items()):
    prefix = f"pred_{position:02d}_{split}"
    save_prediction_bundle(prefix, bundle, split)
    prediction_manifest.append({
        "prefix": prefix, "method": method, "embedding": feature, "split": split,
        "k": (selected_knn_score_k[feature]
              if method == "kNN" and selected_knn_score_k[feature] == selected_knn_cost_k[feature]
              else None),
        "score_k": selected_knn_score_k[feature] if method == "kNN" else None,
        "cost_k": selected_knn_cost_k[feature] if method == "kNN" else None,
    })
pd.DataFrame(prediction_manifest).to_csv(run_dir / "predictions_manifest.csv", index=False)

knn_prediction_manifest = []
for feature, split_predictions in knn_class_predictions.items():
    for split, all_k in split_predictions.items():
        for k, bundle in all_k.items():
            prefix = f"knn_{feature}_k{k}_{split}"
            save_prediction_bundle(prefix, bundle, split)
            knn_prediction_manifest.append({
                "prefix": prefix, "embedding": feature, "split": split, "k": k,
                "selected_for_score": k == selected_knn_score_k[feature],
                "selected_for_cost": k == selected_knn_cost_k[feature],
            })
pd.DataFrame(knn_prediction_manifest).to_csv(run_dir / "knn_predictions_manifest.csv", index=False)

linear_cv_table.to_csv(run_dir / "linear_cv_folds.csv", index=False)
linear_cv_head_table.to_csv(run_dir / "linear_cv_heads.csv", index=False)
linear_cv_summary.to_csv(run_dir / "linear_cv_summary.csv", index=False)
linear_fit_table.to_csv(run_dir / "linear_final_fit.csv", index=False)
joblib.dump(linear_models, run_dir / "linear_models.joblib")
joblib.dump(xgb_models, run_dir / "xgb_models.joblib")

manifest = {
    "mode": RUN_MODE, "seed": SEED, "run_id": run_stamp,
    "execution_provenance": dict(execution_provenance),
    "model_names": model_names, "score_support": score_support.tolist(),
    "label_mapping": "nearest fixed anchor; exact midpoint uses lower anchor",
    "numerical_score_metrics_target": "original recorded score",
    "training_cost_means": cost_means.tolist(),
    "selected_linear_C": selected_linear_C,
    "linear_C_grid": LINEAR_C_GRID.tolist(),
    "linear_CV_numeric_diagnostics": {"outputs": ["hard", "expected"],
        "metrics": ["MSE", "MAE"], "target": "original recorded score",
        "selection": "shared C selected by CV NLL"},
    "linear_C_selection": {"data": "train", "folds": len(training_folds),
                           "metric": "mean out-of-fold NLL across questions and LLMs",
                           "exact_tie_rule": "smaller C"},
    # Record the applied penalties from the newly fitted cost models.
    "cost_ridge_alpha": {feature: float(models["cost_model"].alpha)
                         for feature, models in linear_models.items()},
    "selected_cost_ridge_alpha": selected_cost_ridge_alpha,
    "cost_ridge_cv_metric": "mean per-LLM out-of-fold training Cost WAPE",
    "fit_n_jobs": FIT_N_JOBS, "xgb_parameters": xgb_params, "k_values": K_VALUES,
    "selected_knn_score_k": selected_knn_score_k,
    "selected_knn_cost_k": selected_knn_cost_k,
    "knn_selection": {"data": "train", "folds": len(training_folds),
                      "score_metric": "macro OOF Accuracy",
                      "cost_metric": "macro pooled OOF Cost WAPE",
                      "exact_tie_rule": "larger k"},
    "rows": {split: len(tables[split]["questions"]) for split in SPLITS},
    "test_used": False, "calibration_fitted": False,
    "evaluation_split": EVALUATION_SPLIT,
    "evaluation_purpose": "base-predictor development and diagnostics",
    "score_error_metrics": ["Score_MSE", "Score_MAE"],
    "cost_error_metrics": ["Cost_MSE", "Cost_MAE", "Cost_WAPE"],
    "validation_role": "downstream routing-policy development",
    "versions": {"sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(run_dir / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
# Check core exports here; the final section checks all experiments and marks completion.
assert len(prediction_manifest) == 3 * len(feature_labels) * 2
assert len(knn_prediction_manifest) == len(feature_labels) * 2 * len(K_VALUES)
prediction_prefixes = [row["prefix"] for row in prediction_manifest + knn_prediction_manifest]
assert len(prediction_prefixes) == len(set(prediction_prefixes))
prediction_suffixes = ("_probability.npy", "_expected_score.pkl", "_hard_score.pkl", "_dollar_cost.pkl")
prediction_paths = [run_dir / f"{prefix}{suffix}"
                    for prefix in prediction_prefixes for suffix in prediction_suffixes]
assert all(path.is_file() for path in prediction_paths)
run_status = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
run_status.update({
    "status": "running", "stage": "core_predictors_saved",
    "refit_scope": "all_predictors", "core_prediction_files": len(prediction_paths),
    "rows": {split: len(tables[split]["questions"]) for split in SPLITS},
})
(run_dir / "run_status.json").write_text(json.dumps(run_status, indent=2), encoding="utf-8")

print("Core predictors saved; appended experiments follow. Mode:", RUN_MODE)
print("Fixed score classes:", score_support.tolist())
print("Experiment outputs:", run_dir)
if RUN_MODE == "smoke":
    print("Smoke validates execution. Use full mode for performance comparisons.")


Core predictors saved; appended experiments follow. Mode: full
Fixed score classes: [0.0, 0.25, 0.5, 0.75, 1.0]
Experiment outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full


## 9. Read the results

- Compare accuracy, balanced accuracy and NLL on the same calibration rows.
- Compare hard and expected outputs with raw-score MSE and MAE.
- Inspect per-LLM CSVs when an average hides a weak candidate model.
- Use the training-frequency baseline to assess what the question features add.
- Select C, alpha and k within training, then compare the selected predictors
  on calibration before developing the routing policy on validation.
- Evaluate the selected complete system on the untouched test split.

These are development diagnostics. Earlier exploration repeatedly inspected
validation results; moving the current comparisons to calibration does not
undo that history or create a new independent final estimate.


## 10. Additional cost predictors: LogNormal and Gamma

These experiments adapt the two supplied cost notebooks using this notebook's
prepared inputs, model order and grouped training folds. Each representation
uses one alpha search and one final fit; E5 uses the full 12-value LogNormal
grid directly. The LogNormal E5 final fit explicitly uses alpha 0; the training-CV
winner remains recorded separately. They predict a separate
**positive dollar cost for each of the 11 LLMs**. Existing score classifiers and
their hard/expected-score outputs are reused unchanged.

For LLM $m$, let $c_{im}>0$ be its recorded cost on question $i$, $x_i$ the input
vector, and $N$ the number of rows in the current fitting set. Define
$\mu_m=N^{-1}\sum_{i=1}^{N}c_{im}$ and $r_{im}=c_{im}/\mu_m$.
Every inner, outer and final fit computes its own $\mu_m$ using its fitting rows.
Both representations are evaluated: TF-IDF and E5 with four extra features
(388 dimensions). Fold preprocessing follows the earlier training-only CV.

Three grouped training folds select one alpha per representation by the mean
of the 11 **pooled out-of-fold cost WAPEs**. Exact ties prefer the larger alpha.
Each held-out training row is predicted once per alpha before WAPE is computed;
fold-level WAPE is also saved as a diagnostic. Calibration reports prediction
errors and residuals. Validation cost predictions support later policy selection.
No calibrator or score predictor is fitted in these additional experiments.

The fitting settings match the supplied notebooks: maximum 5,000 iterations,
tolerance $10^{-6}$, and two independent LLM workers during CV. Numerical
libraries use one thread inside scoped fitting blocks. Each family has its own
status file; the final comparison cell completes the whole notebook run.

In [ ]:
# All fitted state is returned by these helpers; the score classifiers stay unchanged.


def costext_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def costext_checked_child(parent, name):
    """Permit a regular direct child, never a symlink, junction or parent escape."""
    parent = Path(parent).resolve()
    if not name or Path(name).name != name or name in {".", ".."}:
        raise ValueError(f"Expected a direct-child output name: {name}")
    path = parent / name
    if path.exists() or path.is_symlink():
        details = path.lstat()
        reparse = getattr(stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0)
        if path.is_symlink() or getattr(details, "st_file_attributes", 0) & reparse:
            raise ValueError(f"Output links are not supported: {path}")
    if path.resolve().parent != parent:
        raise ValueError(f"Output must remain inside {parent}: {path}")
    return path


def costext_path(context, family, suffix):
    if family not in {"lognormal", "gamma"}:
        raise ValueError(f"Unknown cost family: {family}")
    return costext_checked_child(context["run_dir"], f"{family}_{suffix}")


def costext_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


def costext_prediction_specs(family):
    start = {"lognormal": 12, "gamma": 16}[family]
    return [
        (start + offset, feature, split)
        for offset, (feature, split) in enumerate(
            (feature, split)
            for feature in ("tfidf", "e5_388")
            for split in ("calibration", "validation")
        )
    ]


def costext_owned_names(family):
    suffixes = [
        "models.joblib", "predictions_manifest.csv", "experiment_manifest.json",
        "run_status.json", "cv_training_rows.csv", "cv_folds.csv", "cv_summary.csv",
        "cv_per_llm.csv", "final_fit.csv", "calibration_cost_per_llm.csv",
        "calibration_cost_summary.csv", "calibration_residual_summary.csv",
        "calibration_training_mean_baseline.csv",
    ]
    suffixes += [f"{split}_rows.csv" for split in ("train", "calibration", "validation")]
    suffixes += [f"cv_oof_{feature}.npz" for feature in ("tfidf", "e5_388")]
    suffixes += [
        f"residual_length_{feature}_{model:02d}.csv"
        for feature in ("tfidf", "e5_388") for model in range(11)
    ]
    suffixes += [
        f"residual_{feature}_{model:02d}.png"
        for feature in ("tfidf", "e5_388") for model in range(11)
    ]
    names = [f"{family}_{suffix}" for suffix in suffixes]
    names += [
        f"pred_{position:02d}_{split}_dollar_cost.pkl"
        for position, _, split in costext_prediction_specs(family)
    ]
    assert len(names) == len(set(names)) == 66
    return names


def costext_context(*, run_dir, mode, seed, run_id, preparation_dir, table_dir,
                    e5_dir, tables, split_features, model_names, preprocessing,
                    length_scaler, count_scaler, training_folds, train_groups,
                    build_fold_features):
    """Capture canonical inputs without altering the main experiment's variables."""
    root = Path(run_dir).resolve()
    if not root.is_dir():
        raise FileNotFoundError("Run the notebook's configuration and base models first.")
    metadata = json.loads((Path(e5_dir) / "embedding_metadata.json").read_text(encoding="utf-8"))
    names = list(model_names)
    assert len(names) == 11 and len(training_folds) == 3
    groups = np.asarray(train_groups)
    questions = tables["train"]["questions"]
    assert len(groups) == len(questions)
    fold_ids = np.zeros(len(questions), dtype=np.int8)
    folds = []
    for fold_id, (fit_rows, heldout_rows) in enumerate(training_folds, 1):
        fit_rows, heldout_rows = np.asarray(fit_rows), np.asarray(heldout_rows)
        assert not set(groups[fit_rows]).intersection(groups[heldout_rows])
        assert not fold_ids[heldout_rows].any()
        fold_ids[heldout_rows] = fold_id
        folds.append((fit_rows.copy(), heldout_rows.copy()))
    assert (fold_ids > 0).all()
    for split in ("train", "calibration", "validation"):
        frame = tables[split]["costs"].loc[:, names]
        assert frame.index.equals(tables[split]["questions"].index)
        values = frame.to_numpy(dtype=float)
        if not (np.isfinite(values).all() and (values > 0).all()):
            raise ValueError(f"{split}: Gamma and log-cost fitting require positive finite costs.")
        for feature in ("tfidf", "e5_388"):
            assert split_features[feature][split].shape[0] == len(frame)
        assert split_features["e5_388"][split].shape[1] == 388
    preparation = Path(preparation_dir).resolve()
    paths = [Path(table_dir) / "preprocessing.joblib",
             Path(e5_dir) / "embedding_metadata.json",
             Path(e5_dir) / "length_scaler.joblib", Path(e5_dir) / "count_scaler.joblib"]
    for split in ("train", "calibration", "validation"):
        paths += [Path(table_dir) / f"{split}_{name}.pkl"
                  for name in ("questions", "scores", "costs")]
        paths += [Path(table_dir) / f"X_{split}.npz",
                  Path(e5_dir) / f"X_{split}_e5_full_features.npy"]
    input_paths = {path.resolve().relative_to(preparation).as_posix(): path.resolve()
                   for path in paths}
    return {
        "run_dir": root, "mode": mode, "seed": int(seed), "run_id": str(run_id),
        "tables": tables, "split_features": split_features, "model_names": names,
        "preprocessing": preprocessing, "embedding_metadata": metadata,
        "e5_length_scaler": length_scaler, "e5_count_scaler": count_scaler,
        "training_folds": folds, "train_groups": groups.copy(), "fold_ids": fold_ids,
        "build_fold_features": build_fold_features,
        "targets": tables["train"]["costs"].loc[:, names].to_numpy(dtype=float),
        "input_paths": input_paths,
        "input_sha256": {name: costext_digest(path) for name, path in input_paths.items()},
        "feature_labels": {"tfidf": "TF-IDF", "e5_388": "E5 + features (388)"},
        "source_notebooks": {
            "lognormal": {"filename": "Erdos_Project_LogNormal_regression_newest.ipynb",
                "sha256": "bb4d72677359f61b8d2864bcfbe185f5cff8a5202dcbdc8c3089000dff171fc3"},
            "gamma": {"filename": "Erdos_Project_Gamma_regression_newest.ipynb",
                "sha256": "f1f9dccb2589bd839e38db9a7431dd6867bf496c36f8bf3792b8c7ba4c9468ec"},
        },
        "reference_repository_commit": "19c9b696e7e660cc8c9e64bce9fd343905130f67",
    }


def costext_status(context, family, stage, completed=False):
    value = {
        "workflow": family, "mode": context["mode"], "run_id": context["run_id"],
        "status": "completed" if completed else "running", "stage": stage,
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "source_notebook": context["source_notebooks"][family],
        "output_policy": "overwrite_owned_files_preserve_archives",
    }
    costext_json(costext_path(context, family, "run_status.json"), value)


def costext_fit_head(family, X, costs, alpha):
    """Normalize by this fit's mean, and fit the original sklearn estimator."""
    cost_mean = float(np.mean(costs))
    if not np.isfinite(cost_mean) or cost_mean <= 0:
        raise ValueError("The fitting-set cost mean must be positive and finite.")
    if family == "gamma":
        estimator = GammaRegressor(alpha=float(alpha), max_iter=5000, tol=1e-6)
        target = costs / cost_mean
    elif family == "lognormal":
        estimator = Ridge(alpha=float(alpha), solver="lsqr", max_iter=5000, tol=1e-6)
        target = np.log(costs / cost_mean)
    else:
        raise ValueError(f"Unknown cost family: {family}")
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        estimator.fit(X, target)
    iterations = getattr(estimator, "n_iter_", None)
    if iterations is not None and np.max(iterations) >= 5000:
        raise RuntimeError(f"Iteration limit reached for {family}, alpha={alpha}.")
    assert np.isfinite(estimator.coef_).all() and np.isfinite(estimator.intercept_).all()
    return {"family": family, "estimator": estimator, "cost_mean": cost_mean,
            "smearing_factor": 1.0}


def costext_predict_head(head, X):
    value = head["estimator"].predict(X)
    if head["family"] == "gamma":
        prediction = value * head["cost_mean"]
    elif head["family"] == "lognormal":
        prediction = np.exp(value) * head["smearing_factor"] * head["cost_mean"]
    else:
        raise ValueError("The fitted head must identify its cost family.")
    if not (np.isfinite(prediction).all() and (prediction > 0).all()):
        raise ValueError("Cost predictions must be finite and strictly positive.")
    return prediction


def costext_smearing_splits(context, feature, fitting_rows):
    groups = context["train_groups"][fitting_rows]
    inner_cv = GroupKFold(n_splits=3, shuffle=True, random_state=context["seed"])
    splits = []
    for inner_fit, inner_hold in inner_cv.split(fitting_rows, groups=groups):
        fit_rows, heldout_rows = fitting_rows[inner_fit], fitting_rows[inner_hold]
        assert not set(context["train_groups"][fit_rows]).intersection(
            context["train_groups"][heldout_rows])
        X_fit, X_hold = context["build_fold_features"](feature, fit_rows, heldout_rows)
        splits.append((fit_rows, heldout_rows, X_fit, X_hold))
    return splits


def costext_fit_with_smearing(context, family, X, costs, alpha, fitting_rows,
                             inner_splits, all_targets):
    head = costext_fit_head(family, X, costs, alpha)
    if family == "lognormal":
        ratios = np.full(len(fitting_rows), np.nan)
        positions = {int(row): position for position, row in enumerate(fitting_rows)}
        for fit_rows, heldout_rows, X_fit, X_hold in inner_splits:
            inner_head = costext_fit_head(family, X_fit, all_targets[fit_rows], alpha)
            unsmeared_cost = costext_predict_head(inner_head, X_hold)
            inner_positions = [positions[int(row)] for row in heldout_rows]
            ratios[inner_positions] = all_targets[heldout_rows] / unsmeared_cost
        assert np.isfinite(ratios).all() and (ratios > 0).all()
        head["smearing_factor"] = float(ratios.mean())
    return head


def costext_metrics(actual, predicted):
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    error = predicted - actual
    return {
        "Cost_MAE": float(np.abs(error).mean()), "Cost_MSE": float(np.square(error).mean()),
        "Cost_WAPE": float(np.abs(error).sum() / actual.sum()),
        "Gamma_deviance": float(mean_gamma_deviance(actual, predicted)),
        "Relative_mean_bias": float(predicted.mean() / actual.mean() - 1),
    }


def costext_cv_one(context, family, X_fit, X_hold, all_targets, alpha,
                   fitting_rows, inner_splits):
    head = costext_fit_with_smearing(
        context, family, X_fit, all_targets[fitting_rows], alpha,
        fitting_rows, inner_splits, all_targets)
    return costext_predict_head(head, X_hold)


def costext_cv(context, family, alpha_grids, features):
    targets, names = context["targets"], context["model_names"]
    fold_rows, metric_rows, summary_rows = [], [], []
    selected_alpha, selected_oof = {}, {}
    for feature in features:
        grid = np.asarray(alpha_grids[feature], dtype=float)
        oof = np.full((len(grid), len(targets), len(names)), np.nan)
        for fold_id, (fit_rows, heldout_rows) in enumerate(context["training_folds"], 1):
            X_fit, X_hold = context["build_fold_features"](feature, fit_rows, heldout_rows)
            inner = (costext_smearing_splits(context, feature, fit_rows)
                     if family == "lognormal" else [])
            for grid_position, alpha in enumerate(grid):
                started = perf_counter()
                # Limit numerical threads only while this batch is running.
                with threadpool_limits(limits=1):
                    columns = joblib.Parallel(n_jobs=2, backend="threading")(
                        joblib.delayed(costext_cv_one)(
                            context, family, X_fit, X_hold, targets[:, model],
                            float(alpha), fit_rows, inner)
                        for model in range(len(names)))
                oof[grid_position, heldout_rows] = np.column_stack(columns)
                fold_wape = np.mean(
                    np.abs(oof[grid_position, heldout_rows] - targets[heldout_rows]).sum(axis=0)
                    / targets[heldout_rows].sum(axis=0))
                fold_rows.append({
                    "Feature": feature, "Fold": fold_id, "alpha": float(alpha),
                    "Fold_Cost_WAPE": float(fold_wape), "Seconds": perf_counter() - started,
                })
                pd.DataFrame(fold_rows).to_csv(costext_path(context, family, "cv_folds.csv"), index=False)
                print(f"{family}: {feature}, fold {fold_id}/3, alpha={alpha:g}, "
                      f"cost WAPE={fold_wape:.4f}")
            del inner, X_fit, X_hold
        assert np.isfinite(oof).all() and (oof > 0).all()
        for grid_position, alpha in enumerate(grid):
            per_model = [costext_metrics(targets[:, model], oof[grid_position, :, model])
                         for model in range(len(names))]
            metric_rows.extend(
                {"Feature": feature, "Model": name, "alpha": float(alpha), **metrics}
                for name, metrics in zip(names, per_model))
            summary_rows.append({
                "Feature": feature, "alpha": float(alpha),
                **{metric: float(np.mean([row[metric] for row in per_model]))
                   for metric in per_model[0]},
            })
        feature_summary = pd.DataFrame(summary_rows)
        feature_summary = feature_summary.loc[feature_summary["Feature"] == feature]
        best = feature_summary.sort_values(
            ["Cost_WAPE", "alpha"], ascending=[True, False], kind="stable").iloc[0]
        selected_alpha[feature] = float(best["alpha"])
        chosen_position = int(np.flatnonzero(grid == selected_alpha[feature])[0])
        selected_oof[feature] = oof[chosen_position].copy()
        np.savez_compressed(costext_path(context, family, f"cv_oof_{feature}.npz"),
                            alpha=grid, dollar_cost=oof, fold=context["fold_ids"])
        if selected_alpha[feature] in {float(grid.min()), float(grid.max())} and selected_alpha[feature] > 0:
            print(f"Selected alpha is on the searched boundary: {family}, {feature}, "
                  f"alpha={selected_alpha[feature]:g}.")
        del oof
    return {"cv_folds": pd.DataFrame(fold_rows), "cv_per_llm": pd.DataFrame(metric_rows),
            "cv_summary": pd.DataFrame(summary_rows), "selected_alpha": selected_alpha,
            "selected_oof": selected_oof}


def costext_final_fit(context, family, fit_alpha, features):
    models, predictions, fit_rows = {}, {}, []
    targets, names = context["targets"], context["model_names"]
    fitting_rows = np.arange(len(targets))
    with threadpool_limits(limits=1):
        for feature in features:
            # These folds cover the full training set; neither development split enters smearing.
            inner = []
            if family == "lognormal":
                for fit_rows_inner, heldout_rows in context["training_folds"]:
                    X_fit, X_hold = context["build_fold_features"](
                        feature, fit_rows_inner, heldout_rows)
                    inner.append((fit_rows_inner, heldout_rows, X_fit, X_hold))
            models[feature] = []
            alpha = fit_alpha[feature]
            for model, name in enumerate(names):
                started = perf_counter()
                head = costext_fit_with_smearing(
                    context, family, context["split_features"][feature]["train"],
                    targets[:, model], alpha, fitting_rows, inner, targets[:, model])
                models[feature].append(head)
                fit_rows.append({"Feature": feature, "Model": name, "alpha": float(alpha),
                    "smearing_factor": head["smearing_factor"], "cost_mean": head["cost_mean"],
                    "Seconds": perf_counter() - started})
            for split in ("calibration", "validation"):
                values = np.column_stack([
                    costext_predict_head(head, context["split_features"][feature][split])
                    for head in models[feature]])
                predictions[(feature, split)] = pd.DataFrame(
                    values, index=context["tables"][split]["questions"].index, columns=names)
            del inner
    return {"models": models, "predictions": predictions, "final_fit": pd.DataFrame(fit_rows)}


def costext_diagnostics(context, family, result):
    metrics, residual_stats = [], []
    actual = context["tables"]["calibration"]["costs"].loc[:, context["model_names"]]
    lengths = context["tables"]["calibration"]["questions"]["question_text"].str.len().to_numpy()
    for feature in ("tfidf", "e5_388"):
        predicted = result["predictions"][(feature, "calibration")]
        assert actual.index.equals(predicted.index)
        assert predicted.columns.tolist() == context["model_names"]
        for model, name in enumerate(context["model_names"]):
            y, p = actual[name].to_numpy(dtype=float), predicted[name].to_numpy(dtype=float)
            metrics.append({"Feature": feature, "Model": name, **costext_metrics(y, p)})
            head = result["models"][feature][model]
            if family == "lognormal":
                residual = np.log(y / head["cost_mean"]) - head["estimator"].predict(
                    context["split_features"][feature]["calibration"])
            else:
                residual = (y - p) / p
            residual_stats.append({
                "Feature": feature, "Model": name, "mean": float(residual.mean()),
                "std": float(residual.std()), "skewness": float(skew(residual)),
                "excess_kurtosis": float(kurtosis(residual)),
            })
            frame = pd.DataFrame({"residual": residual, "length": lengths})
            frame["length_bin"] = pd.qcut(frame["length"], 5, duplicates="drop")
            grouped = frame.groupby("length_bin", observed=True)["residual"].agg(["count", "mean", "std"])
            grouped.to_csv(costext_path(context, family, f"residual_length_{feature}_{model:02d}.csv"))
            fig, axes = plt.subplots(1, 2, figsize=(10, 3))
            try:
                axes[0].hist(residual, bins=50)
                axes[0].set_title("Calibration residuals")
                if family == "lognormal":
                    probplot(residual, dist="norm", plot=axes[1])
                    axes[1].set_title("Normal Q-Q diagnostic")
                else:
                    axes[1].scatter(p, residual, s=3, alpha=.3)
                    axes[1].set_xlabel("Predicted dollar cost")
                    axes[1].set_ylabel("(actual - predicted) / predicted")
                fig.suptitle(f"{feature} — {name}")
                fig.tight_layout()
                fig.savefig(costext_path(context, family, f"residual_{feature}_{model:02d}.png"), dpi=130)
            finally:
                plt.close(fig)
    result["calibration_per_llm"] = pd.DataFrame(metrics)
    result["calibration_summary"] = result["calibration_per_llm"].groupby(
        "Feature", sort=False).mean(numeric_only=True)
    result["calibration_residual_summary"] = pd.DataFrame(residual_stats)
    result["calibration_training_mean_baseline"] = pd.DataFrame([
        {"Model": name, **costext_metrics(actual[name].to_numpy(dtype=float),
            np.full(len(actual), context["targets"][:, model].mean()))}
        for model, name in enumerate(context["model_names"])])
    return result


def costext_export(context, family, result, *, stage):
    names = context["model_names"]
    prediction_rows = []
    for position, feature, split in costext_prediction_specs(family):
        frame = result["predictions"][(feature, split)]
        assert frame.columns.tolist() == names
        assert frame.index.equals(context["tables"][split]["questions"].index)
        assert np.isfinite(frame.to_numpy()).all() and (frame.to_numpy() > 0).all()
        prefix = f"pred_{position:02d}_{split}"
        frame.to_pickle(costext_checked_child(context["run_dir"], f"{prefix}_dollar_cost.pkl"))
        prediction_rows.append({"prefix": prefix, "method": family, "embedding": feature,
                                "split": split, "bundle_type": "cost_only"})
    pd.DataFrame(prediction_rows).to_csv(costext_path(context, family, "predictions_manifest.csv"), index=False)
    rule = ("estimator.predict(X) * cost_mean" if family == "gamma" else
            "exp(estimator.predict(X)) * smearing_factor * cost_mean")
    joblib.dump({"family": family, "models": result["models"], "model_names": names,
        "preprocessing": context["preprocessing"], "embedding_metadata": context["embedding_metadata"],
        "e5_length_scaler": context["e5_length_scaler"], "e5_count_scaler": context["e5_count_scaler"],
        "prediction_rule": rule, "run_id": context["run_id"],
        "selected_alpha": result["selected_alpha"],
        "cv_selected_alpha": result["selected_alpha"], "fit_alpha": result["fit_alpha"],
        "final_fit_overrides": result["final_fit_overrides"]},
        costext_path(context, family, "models.joblib"))
    for split in ("train", "calibration", "validation"):
        context["tables"][split]["questions"][["sample_id"]].to_csv(
            costext_path(context, family, f"{split}_rows.csv"))
    training_rows = context["tables"]["train"]["questions"][["sample_id", "question_group"]].copy()
    training_rows["Fold"] = context["fold_ids"]
    training_rows.to_csv(costext_path(context, family, "cv_training_rows.csv"))
    for key in ("cv_folds", "cv_summary", "cv_per_llm", "final_fit", "calibration_residual_summary",
                "calibration_training_mean_baseline"):
        result[key].to_csv(costext_path(context, family, f"{key}.csv"), index=False)
    result["calibration_per_llm"].to_csv(costext_path(context, family, "calibration_cost_per_llm.csv"), index=False)
    result["calibration_summary"].to_csv(costext_path(context, family, "calibration_cost_summary.csv"))
    manifest = {
        "workflow": family, "mode": context["mode"], "seed": context["seed"],
        "run_id": context["run_id"], "stage": stage,
        "reference_repository_commit": context["reference_repository_commit"],
        "source_notebook": context["source_notebooks"][family],
        "evaluation_split": "calibration", "validation_role": "routing-policy development only",
        "test_used": False, "calibration_fitted": False,
        "prior_experiments": "Original collaborator notebooks previously inspected validation and test.",
        "feature_labels": context["feature_labels"], "model_names": names,
        "rows": {split: len(context["tables"][split]["questions"])
                 for split in ("train", "calibration", "validation")},
        "selected_alpha": result["selected_alpha"],
        "cv_selected_alpha": result["selected_alpha"], "fit_alpha": result["fit_alpha"],
        "final_fit_overrides": result["final_fit_overrides"],
        "alpha_grids": result["alpha_grids"],
        "fit_n_jobs": 2, "worker_backend": "threading", "numerical_library_threads": 1,
        "max_iter": 5000, "tol": 1e-6,
        "cv": "three shuffled GroupKFold folds; canonical seed and fold-local preprocessing",
        "cost_selection": "macro per-LLM pooled OOF Cost WAPE; exact ties prefer larger alpha",
        "final_fit_policy": "fit_alpha uses CV-selected alpha except for recorded explicit overrides",
        "normalization": "each fitting set's per-LLM cost mean; predictions restored to dollars",
        "smearing": ("training-only grouped OOF residual ratios, nested inside outer CV"
                     if family == "lognormal" else None),
        "cost_only": True, "score_predictors_fitted": False, "prediction_rule": rule,
        "output_policy": "overwrite_owned_files_preserve_archives",
        "input_sha256": context["input_sha256"],
        "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    own_paths = [costext_checked_child(context["run_dir"], name) for name in costext_owned_names(family)]
    manifest["output_files"] = [path.name for path in own_paths]
    result["manifest_path"] = costext_path(context, family, "experiment_manifest.json")
    result["status_path"] = costext_path(context, family, "run_status.json")
    costext_json(result["manifest_path"], manifest)
    costext_status(context, family, stage, completed=True)
    assert all(path.is_file() and path.stat().st_size > 0 for path in own_paths)
    result["output_paths"] = own_paths
    result["family"] = family
    result["manifest"] = manifest
    return result


def costext_run_stage(context, family, alpha_grids, *, fit_alpha_overrides=None,
                      fit_alpha_override_reason=None):
    """Keep training-CV choices and explicit final-fit alpha choices separate."""
    features = ("tfidf", "e5_388")
    if set(alpha_grids) != set(features):
        raise ValueError("Provide one alpha grid for TF-IDF and one for E5.")
    grids = {}
    for feature in features:
        grid = np.asarray(alpha_grids[feature], dtype=float)
        if (grid.ndim != 1 or grid.size == 0 or not np.isfinite(grid).all()
                or (grid < 0).any() or len(np.unique(grid)) != len(grid)):
            raise ValueError(f"Invalid alpha grid for {feature}: {alpha_grids[feature]}")
        grids[feature] = grid.tolist()
    overrides = dict(fit_alpha_overrides or {})
    if not set(overrides).issubset(features):
        raise ValueError(f"Unknown final-fit override features: {set(overrides) - set(features)}")
    for feature, alpha in overrides.items():
        value = float(alpha)
        if not np.isfinite(value) or value < 0:
            raise ValueError(f"Invalid final-fit alpha override for {feature}: {alpha}")
        overrides[feature] = value
    reason = (str(fit_alpha_override_reason) if fit_alpha_override_reason is not None else
              "Explicit final-fit alpha override; training CV outcome is retained.") if overrides else None
    costext_status(context, family, "cross-validation")
    result = costext_cv(context, family, grids, features)
    result["alpha_grids"] = grids
    result["cv_selected_alpha"] = dict(result["selected_alpha"])
    result["fit_alpha"] = {**result["selected_alpha"], **overrides}
    result["final_fit_overrides"] = {
        feature: {"alpha": alpha, "reason": reason} for feature, alpha in overrides.items()
    }
    costext_status(context, family, "final fitting")
    result.update(costext_final_fit(context, family, result["fit_alpha"], features))
    costext_status(context, family, "calibration diagnostics and export")
    result = costext_diagnostics(context, family, result)
    result = costext_export(context, family, result, stage="complete")
    display(result["cv_summary"])
    display(result["calibration_summary"])
    return result


costext_shared = costext_context(
    run_dir=run_dir, mode=RUN_MODE, seed=SEED, run_id=run_stamp,
    preparation_dir=preparation_dir, table_dir=table_dir, e5_dir=e5_dir,
    tables=tables, split_features=split_features, model_names=model_names,
    preprocessing=preprocessing, length_scaler=e5_length_scaler,
    count_scaler=e5_count_scaler, training_folds=training_folds,
    train_groups=train_groups, build_fold_features=build_fold_features,
)

### 10.1 LogNormal regression and training-only smearing

Let $\beta_{0m}$ be an intercept, $\beta_m$ a coefficient vector, and $\alpha\ge0$
the L2 penalty. Ridge regression fits normalized **log cost**:

$$
\widehat\eta_m(x)=\beta_{0m}+x^\top\beta_m,\qquad
\min_{\beta_{0m},\beta_m}
\sum_{i=1}^{N}\left[\log r_{im}-\beta_{0m}-x_i^\top\beta_m\right]^2
+\alpha\|\beta_m\|_2^2.
$$

Exponentiation gives a prediction in normalized cost units. Smearing corrects
its retransformation using $S_m$, the mean of observed cost divided by an
**unsmeared out-of-fold cost prediction** on the fitting set:

$$
S_m=\frac1N\sum_{i=1}^{N}\frac{c_{im}}{\widetilde c_{im}^{\mathrm{OOF}}},
\qquad
\widehat c_m(x)=\mu_m S_m\exp\!\left(\widehat\eta_m(x)\right).
$$

Here $\widetilde c_{im}^{\mathrm{OOF}}$ comes from an inner three-fold grouped
fit that excludes question $i$'s group. For each outer CV fit, this inner CV uses
only the outer training portion. The final fit estimates $S_m$ using three folds
of the full training set. Each inner model restores its own fitting-set mean
before its dollar prediction enters the ratio. This preserves the supplied
procedure without assuming normally distributed residuals to choose $S_m$.
A single smearing factor is a mean-cost approximation; it does not guarantee
an unbiased conditional mean at every input.

Run one grouped three-fold CV search for each representation:

| Representation | Candidate alpha values |
|---|---|
| TF-IDF | 0.01, 0.1, 1, 10, 100, 1000, 10000 |
| E5 + features (388) | 0, 0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000 |

For each candidate, pool its three outer-fold predictions before computing each
LLM's WAPE. Select one alpha per representation by the mean WAPE across the 11
LLMs; exact ties favor the larger alpha. `selected_alpha` and
`cv_selected_alpha` retain this training-CV result. TF-IDF uses that alpha for its
11 final LLM heads. The E5 final model explicitly overrides the CV choice with
**alpha 0**, shared by its 11 LLM heads. This user-requested final-fit choice is
recorded in `fit_alpha` and `final_fit_overrides`; it does not change the CV
tables or the selected CV winner. A complete rerun retains the explicit override.

Each representation is fitted on the full training set, including a
training-only OOF estimate of its smearing factors at its actual `fit_alpha`.
Calibration and validation labels do not select alpha.

The saved model contains coefficients, intercept, training cost mean and
smearing factor for every LLM.

In [ ]:
lognormal_alpha_grids = {
    "tfidf": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0],
    "e5_388": [0.0, 1e-6, 1e-5, 1e-4, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0],
}
# Preserve the CV winner; the requested E5 final model uses unregularized log-cost fitting.
ln_result = costext_run_stage(
    costext_shared, "lognormal", lognormal_alpha_grids,
    fit_alpha_overrides={"e5_388": 0.0},
    fit_alpha_override_reason="Match the collaborator final-fit setting; retain the training-CV alpha separately.",
)


Partial update 20261028T091427_279951Z_lognormal_e5_alpha0: only LogNormal E5 final/smearing fits rerun at alpha=0; CV and other predictors retained.


,Feature,alpha,Cost_MAE,Cost_MSE,Cost_WAPE,Gamma_deviance,Relative_mean_bias
0,tfidf,0.010000,0.000153,3.655093e-07,0.165892,0.047603,0.010955
1,tfidf,0.100000,0.000129,3.330701e-07,0.136421,0.032496,-0.003591
2,tfidf,1.000000,0.000125,3.515309e-07,0.130008,0.029974,-0.010401
3,tfidf,10.000000,0.000141,4.144233e-07,0.146729,0.039418,-0.017891
4,tfidf,100.000000,0.000191,5.524770e-07,0.203773,0.092724,-0.036247
5,tfidf,1000.000000,0.000327,8.812136e-07,0.363346,0.253867,-0.072546
6,tfidf,10000.000000,0.000530,1.593157e-06,0.563808,0.488346,-0.026351
7,e5_388,0.000000,0.000123,3.409607e-07,0.126639,0.031540,0.000831
8,e5_388,0.000001,0.000123,3.409620e-07,0.126639,0.031540,0.000831
9,e5_388,0.000010,0.000123,3.409739e-07,0.126640,0.031540,0.000832


,Cost_MAE,Cost_MSE,Cost_WAPE,Gamma_deviance,Relative_mean_bias
Feature,,,,,
tfidf,0.000126,3.828792e-07,0.129741,0.029123,-0.011921
e5_388,0.000129,5.236276e-07,0.134409,0.031244,0.010577


## 11. Gamma regression with a log link

Gamma regression models positive normalized cost directly. For LLM $m$, its
predicted normalized mean is $\widehat r_m(x)=\exp(\beta_{0m}+x^\top\beta_m)$.
Scikit-learn's Gamma loss with L2 regularization is

$$
\min_{\beta_{0m},\beta_m}
\frac{1}{2N}\sum_{i=1}^{N}d_\Gamma(r_{im},\widehat r_m(x_i))
+\frac{\alpha}{2}\|\beta_m\|_2^2,
\qquad
d_\Gamma(y,p)=2\left[\log\!\left(\frac p y\right)+\frac y p-1\right].
$$

Here $d_\Gamma$ is Gamma deviance for positive observed value $y$ and prediction
$p$. The log link guarantees a positive predicted mean. Restore dollars with
$\widehat c_m(x)=\mu_m\widehat r_m(x)$. The supplied search uses
$\alpha\in\{10^{-6},10^{-5},10^{-4},10^{-3},10^{-2},0.1,1\}$, selected by the
same pooled OOF cost WAPE. This family has no smearing step. Both families export
point estimates of mean cost; no cost probability distribution is exported.

In [21]:
gamma_alpha_grids = {
    feature: [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0]
    for feature in ("tfidf", "e5_388")
}
gamma_result = costext_run_stage(costext_shared, "gamma", gamma_alpha_grids)


gamma: tfidf, fold 1/3, alpha=1e-06, cost WAPE=0.1575


gamma: tfidf, fold 1/3, alpha=1e-05, cost WAPE=0.1343


gamma: tfidf, fold 1/3, alpha=0.0001, cost WAPE=0.1324


gamma: tfidf, fold 1/3, alpha=0.001, cost WAPE=0.1515


gamma: tfidf, fold 1/3, alpha=0.01, cost WAPE=0.2192


...

gamma: e5_388, fold 3/3, alpha=0.01, cost WAPE=0.1537


gamma: e5_388, fold 3/3, alpha=0.1, cost WAPE=0.1752


gamma: e5_388, fold 3/3, alpha=1, cost WAPE=0.2825


,Feature,alpha,Cost_MAE,Cost_MSE,Cost_WAPE,Gamma_deviance,Relative_mean_bias
0,tfidf,0.000001,0.000145,3.562865e-07,0.156482,0.044437,-0.014156
1,tfidf,0.000010,0.000126,3.341488e-07,0.133253,0.031578,-0.016054
2,tfidf,0.000100,0.000126,3.535555e-07,0.131005,0.030075,-0.013844
3,tfidf,0.001000,0.000143,4.139048e-07,0.150214,0.041205,-0.021655
4,tfidf,0.010000,0.000201,5.659495e-07,0.219395,0.106245,-0.059359
5,tfidf,0.100000,0.000377,1.048304e-06,0.415985,0.300864,-0.090341
6,tfidf,1.000000,0.000540,1.648248e-06,0.575559,0.507629,-0.029507
7,e5_388,0.000001,0.000128,3.333355e-07,0.132536,0.030986,0.003448
8,e5_388,0.000010,0.000127,3.339473e-07,0.131966,0.031014,0.003122
9,e5_388,0.000100,0.000126,3.445220e-07,0.130613,0.032153,0.002215


,Cost_MAE,Cost_MSE,Cost_WAPE,Gamma_deviance,Relative_mean_bias
Feature,,,,,
tfidf,0.000128,3.898691e-07,0.132046,0.029770,-0.015355
e5_388,0.000134,5.889103e-07,0.142286,0.032954,0.014560


### 11.1 Cost diagnostics and output handoff

For one LLM, let $n$ be the number of calibration questions, $c_i>0$ its recorded
costs, and $p_i>0$ its predictions. The reported metrics are

$$
\mathrm{MAE}=\frac1n\sum_i|p_i-c_i|,\qquad
\mathrm{MSE}=\frac1n\sum_i(p_i-c_i)^2,\qquad
\mathrm{WAPE}=\frac{\sum_i|p_i-c_i|}{\sum_i c_i},
$$

$$
\mathrm{GammaDeviance}=\frac1n\sum_i d_\Gamma(c_i,p_i),\qquad
\mathrm{RelativeMeanBias}=\frac{n^{-1}\sum_i p_i}{n^{-1}\sum_i c_i}-1.
$$

Lower MAE, MSE, WAPE and Gamma deviance indicate better predictions; mean bias
is best near zero and its sign shows overprediction or underprediction.
Summary rows average per-LLM metrics with equal weights. Training-mean cost
predictions provide the same constant reference for both representations.

For LogNormal, residuals are $\log(c_i/\mu_m)-\widehat\eta_m(x_i)$, measured
before smearing. For Gamma, residuals are $(c_i-p_i)/p_i$. Each family saves a
residual histogram plus a Normal Q-Q diagnostic (LogNormal) or a residual-versus-
prediction scatter plot (Gamma), for each representation and LLM: **22 PNGs**.
It also saves 22 CSVs grouping residuals into five question-length quantile bins.
Residual summaries contain mean, standard deviation, skewness (asymmetry), and
excess kurtosis (tail weight relative to a Normal distribution). A Normal Q-Q
plot compares observed residual quantiles with Normal quantiles; deviations
from a line reveal departures from Normal shape. These diagnostics do not fit
a calibrator or choose alpha.

All outputs share the existing run directory. The original complete score/cost
manifest stays unchanged. `lognormal_predictions_manifest.csv` and
`gamma_predictions_manifest.csv` identify the eight additional cost frames as
`cost_only`. Their fixed file numbers are:

| Family | TF-IDF calibration / validation | E5 calibration / validation |
|---|---|---|
| LogNormal | `pred_12_calibration_dollar_cost.pkl` / `pred_13_validation_dollar_cost.pkl` | `pred_14_calibration_dollar_cost.pkl` / `pred_15_validation_dollar_cost.pkl` |
| Gamma | `pred_16_calibration_dollar_cost.pkl` / `pred_17_validation_dollar_cost.pkl` | `pred_18_calibration_dollar_cost.pkl` / `pred_19_validation_dollar_cost.pkl` |

All other additional files use `lognormal_` or `gamma_` prefixes, including
`*_models.joblib`, CV tables, OOF arrays, calibration tables and run manifests.
The OOF `.npz` files contain `alpha`, `dollar_cost` (alpha × question × LLM) and
`fold`. Model bundles include standard sklearn estimators, means, smearing
factors where applicable, preprocessing objects and E5 scalers.

For routing, pair one new cost frame with an existing hard/expected-score frame
from the same representation, questions and LLM order. Check indices and columns
before pairing and select the resulting policy on validation. The final summary
below compares cost-prediction errors; routing cost–quality trade-offs require
that separate policy evaluation. Fractional benchmark scores measure graded
quality and are not binary correctness probabilities.

## 12. Compare the final predictors

The score comparison contains the three classifiers—L2 Logistic, kNN and
XGBoost—with each input representation, plus the training-frequency
baseline. Classification metrics evaluate the same five-class
probabilities. Hard and expected scores share those probabilities; their
numerical MSE and MAE are evaluated separately against the original
recorded scores. Gamma and LogNormal predict cost only, so they have no
score-classification results.

The cost comparison contains Ridge, kNN, XGBoost, LogNormal and Gamma
for both representations. A training-mean cost baseline predicts the
same per-LLM training mean for every question. Every method is evaluated
on the same calibration rows and the same 11 LLMs, using dollar costs.
LogNormal E5 uses the explicit final-fit alpha 0 override. Its training-CV
selection remains a separate record in the family manifest and CV tables.

Let \(c_{im}>0\) be the recorded cost for calibration question \(i\)
and LLM \(m\), and let \(\widehat c_{im}\) be its predicted cost. With
\(n\) calibration questions, the per-LLM errors are

\[
\operatorname{MAE}_m=\frac1n\sum_{i=1}^{n}|\widehat c_{im}-c_{im}|,
\qquad
\operatorname{MSE}_m=\frac1n\sum_{i=1}^{n}(\widehat c_{im}-c_{im})^2,
\]

\[
\operatorname{WAPE}_m=
\frac{\sum_{i=1}^{n}|\widehat c_{im}-c_{im}|}
     {\sum_{i=1}^{n}c_{im}}.
\]

Each summary column is the arithmetic mean of its 11 per-LLM values.
This gives every LLM equal weight. MAE is in dollars, MSE is in squared
dollars, and WAPE is displayed as a percentage. Lower is better for all
three. The tables are calculated from the predictions produced in this
run; values from the collaborators' saved runs are not imported.

These tables measure prediction quality. Cost savings at a chosen
performance level require a routing policy, which will be developed
using the exported validation predictions.


In [ ]:
assert EVALUATION_SPLIT == "calibration"
cost_comparison_truth = tables["calibration"]["costs"].loc[:, model_names]
assert cost_comparison_truth.index.equals(tables["calibration"]["questions"].index)
assert cost_comparison_truth.columns.tolist() == model_names
cost_comparison_actual = cost_comparison_truth.to_numpy(dtype=float)
assert np.isfinite(cost_comparison_actual).all()
assert (cost_comparison_actual > 0).all()

# The linear score classifier uses Ridge for its separate cost target.
cost_comparison_source_names = {
    "L2 logistic": "Ridge", "kNN": "kNN", "XGBoost": "XGBoost",
}
cost_comparison_frames = {
    (cost_method, feature): prediction_bundles[(source_method, feature, "calibration")]["cost"]
    for source_method, cost_method in cost_comparison_source_names.items()
    for feature in feature_labels
}
for cost_method, cost_result in (("LogNormal", ln_result), ("Gamma", gamma_result)):
    for feature in feature_labels:
        cost_comparison_frames[(cost_method, feature)] = cost_result["predictions"][(feature, "calibration")]

cost_comparison_training_means = tables["train"]["costs"].loc[:, model_names].mean().to_numpy(dtype=float)
cost_comparison_frames[("Training mean", "Baseline")] = pd.DataFrame(
    np.broadcast_to(cost_comparison_training_means, cost_comparison_actual.shape),
    index=cost_comparison_truth.index, columns=model_names,
)

def cost_comparison_errors(predicted):
    """Return aligned dollar-error diagnostics for each candidate LLM."""
    assert isinstance(predicted, pd.DataFrame)
    assert predicted.index.equals(cost_comparison_truth.index)
    assert predicted.columns.tolist() == model_names
    values = predicted.to_numpy(dtype=float)
    assert values.shape == cost_comparison_actual.shape
    assert np.isfinite(values).all() and (values > 0).all()
    difference = values - cost_comparison_actual
    absolute_error = np.abs(difference)
    return pd.DataFrame({
        "Cost_MAE": absolute_error.mean(axis=0),
        "Cost_MSE": np.square(difference).mean(axis=0),
        "Cost_WAPE": absolute_error.sum(axis=0) / cost_comparison_actual.sum(axis=0),
    }, index=pd.Index(model_names, name="model"))

cost_comparison_per_llm = pd.concat({
    key: cost_comparison_errors(frame)
    for key, frame in cost_comparison_frames.items()
}, names=["cost_method", "embedding"])
cost_comparison_summary = cost_comparison_per_llm.groupby(
    level=["cost_method", "embedding"], sort=False,
).mean()
assert len(cost_comparison_summary) == 5 * len(feature_labels) + 1
assert np.isfinite(cost_comparison_summary.to_numpy(dtype=float)).all()

cost_comparison_paths = [
    run_dir / "calibration_cost_comparison_per_llm.csv",
    run_dir / "calibration_cost_comparison_summary.csv",
]
cost_comparison_per_llm.to_csv(cost_comparison_paths[0])
cost_comparison_summary.to_csv(cost_comparison_paths[1])

print("Calibration classification: three classifier families with two representations")
display(distribution_summary.style.format({
    "Accuracy": "{:.1%}", "Balanced_accuracy": "{:.1%}",
    "NLL": "{:.4f}", "Zero_true_probability_rate": "{:.1%}",
}))
print("Calibration numerical score errors: hard class and expected score")
display(numeric_summary.loc[:, ["Score_MSE", "Score_MAE"]].style.format("{:.5f}"))
print("Calibration dollar-cost errors: five cost families with two representations")
display(cost_comparison_summary.style.format({
    "Cost_MAE": "${:.8f}", "Cost_MSE": "{:.3e}", "Cost_WAPE": "{:.2%}",
}))


Calibration classification: retained from the completed full run


Accuracy  Balanced_accuracy       NLL  Zero_true_probability_rate
method             embedding                                                                   
Training frequency Baseline   0.535661           0.200000  1.180041                    0.000000
L2 logistic        tfidf      0.686952           0.452885  0.637180                    0.000000
                   e5_388     0.690316           0.449896  0.634146                    0.000000
kNN                tfidf      0.678280           0.436115  0.713428                    0.002243
                   e5_388     0.686279           0.449428  0.675153                    0.001744
XGBoost            tfidf      0.691313           0.448097  0.642598                    0.000000
                   e5_388     0.690765           0.443018  0.654871                    0.000000

Calibration numerical score errors: retained from the completed full run


Score_MSE  Score_MAE
method             embedding score_output                        
Training frequency Baseline  Expected score   0.178830   0.382129
                             Hard class       0.303103   0.344263
L2 logistic        tfidf     Expected score   0.155706   0.322452
                             Hard class       0.242585   0.261887
                   e5_388    Expected score   0.154270   0.329567
                             Hard class       0.239498   0.258713
kNN                tfidf     Expected score   0.156523   0.330458
                             Hard class       0.243162   0.264995
                   e5_388    Expected score   0.155116   0.330628
                             Hard class       0.241195   0.261063
XGBoost            tfidf     Expected score   0.154136   0.333373
                             Hard class       0.238387   0.257622
                   e5_388    Expected score   0.155984   0.339072
                             Hard class       0.239278   0.258418

Calibration dollar-cost errors: LogNormal E5 updated at final-fit alpha=0


Cost_MAE      Cost_MSE  Cost_WAPE
cost_method   embedding                                   
Ridge         tfidf      0.000168  4.211060e-07   0.172387
              e5_388     0.000146  3.679821e-07   0.134896
kNN           tfidf      0.000211  5.764476e-07   0.234120
              e5_388     0.000099  2.227246e-07   0.104226
XGBoost       tfidf      0.000121  2.843570e-07   0.126522
              e5_388     0.000118  3.002898e-07   0.121769
LogNormal     tfidf      0.000126  3.828792e-07   0.129741
              e5_388     0.000129  5.236276e-07   0.134409
Gamma         tfidf      0.000128  3.898691e-07   0.132046
              e5_388     0.000134  5.889103e-07   0.142286
Training mean Baseline   0.000589  1.872519e-06   0.614599

## 13. Check the saved run and hand off the predictions

`predictions_manifest.csv` continues to describe the 12 main
score-and-cost bundles; `knn_predictions_manifest.csv` describes the
40 candidate-k bundles. Each bundle has four prediction files, giving
208 core prediction files. The additional cost predictors export eight
dollar-cost files, identified by `lognormal_predictions_manifest.csv`
and `gamma_predictions_manifest.csv`. The current run therefore has
216 active prediction files. Each additional cost family also saves
its CV diagnostics, fitted models and calibration residual diagnostics.

The two additional manifests mark their bundles as `cost_only`.
Downstream code may pair these costs with any score predictor after
checking the split, question index and LLM column order. A change of
cost predictor requires the routing-policy settings to be selected
again on validation.

The final check below verifies the declared files, the preserved training-CV
winner, and each final estimator's actual `fit_alpha`, then records
completion in `run_status.json`. Family manifests and model bundles record
`selected_alpha`/`cv_selected_alpha` separately from `fit_alpha` and explicit
override metadata. It checks the current run's files;
archived results are kept for reference. Calibration summaries remain
development diagnostics. Earlier inspection of validation results is
part of the project's history; the current split roles do not remove
that history. Final claims about savings should come from evaluating
the selected complete routing system.


In [ ]:
# Validate the files declared by this run before reporting completion.
completion_root = run_dir.resolve()

def completion_path(value):
    candidate = Path(value)
    if not candidate.is_absolute():
        candidate = run_dir / candidate
    candidate = candidate.resolve()
    assert candidate.is_relative_to(completion_root), f"Artifact outside run directory: {candidate}"
    assert candidate.is_file() and candidate.stat().st_size > 0, f"Missing or empty artifact: {candidate}"
    return candidate

assert len(prediction_manifest) == 12
assert len(knn_prediction_manifest) == 40
assert len(prediction_paths) == 208
completion_core_predictions = [completion_path(path) for path in prediction_paths]
assert len(set(completion_core_predictions)) == 208

completion_required_core = [
    "predictions_manifest.csv", "knn_predictions_manifest.csv",
    "linear_models.joblib", "xgb_models.joblib",
    "linear_cv_folds.csv", "linear_cv_heads.csv", "linear_cv_summary.csv", "linear_final_fit.csv",
    "raw_score_anchor_coverage.csv", "raw_score_value_counts.csv",
    "train_rows.csv", "calibration_rows.csv", "validation_rows.csv",
    "calibration_classification_summary.csv", "calibration_classification_per_llm.csv",
    "calibration_numeric_summary.csv", "calibration_numeric_per_llm.csv",
    "cost_ridge_cv/cost_cv_folds.csv", "cost_ridge_cv/cost_cv_manifest.json",
    "cost_ridge_cv/cost_cv_oof_e5_388.npz", "cost_ridge_cv/cost_cv_oof_tfidf.npz",
    "cost_ridge_cv/cost_cv_per_llm.csv", "cost_ridge_cv/cost_cv_summary.csv",
    "cost_ridge_cv/cost_cv_training_rows.csv",
    "knn_cv/knn_cv_folds.csv", "knn_cv/knn_cv_manifest.json",
    "knn_cv/knn_cv_oof_e5_388.npz", "knn_cv/knn_cv_oof_tfidf.npz",
    "knn_cv/knn_cv_per_llm.csv", "knn_cv/knn_cv_selection.csv", "knn_cv/knn_cv_summary.csv",
    "knn_cv/knn_cv_training_rows.csv", "experiment_manifest.json", "run_status.json",
]
completion_artifacts = completion_core_predictions + [
    completion_path(name) for name in completion_required_core
] + [completion_path(path) for path in cost_comparison_paths]
completion_cost_predictions = []
completion_extension_records = {}
completion_expected_rows = {split: len(tables[split]["questions"]) for split in SPLITS}

for completion_family, completion_label, completion_result, completion_start in (
    ("lognormal", "LogNormal", ln_result, 12), ("gamma", "Gamma", gamma_result, 16),
):
    completion_manifest_path = completion_path(f"{completion_family}_predictions_manifest.csv")
    completion_prediction_manifest = pd.read_csv(completion_manifest_path, dtype={"prefix": str})
    completion_manifest_columns = {"prefix", "method", "embedding", "split", "bundle_type"}
    assert completion_manifest_columns.issubset(completion_prediction_manifest.columns)
    assert len(completion_prediction_manifest) == 4
    assert completion_prediction_manifest["prefix"].is_unique
    assert completion_prediction_manifest["bundle_type"].eq("cost_only").all()
    completion_expected_pairs = [
        (feature, split) for feature in ("tfidf", "e5_388")
        for split in ("calibration", "validation")
    ]
    completion_expected_prefixes = {
        (feature, split): f"pred_{completion_start + offset:02d}_{split}"
        for offset, (feature, split) in enumerate(completion_expected_pairs)
    }
    assert set(zip(completion_prediction_manifest["embedding"], completion_prediction_manifest["split"])) == set(completion_expected_pairs)
    for completion_row in completion_prediction_manifest.itertuples(index=False):
        completion_key = (completion_row.embedding, completion_row.split)
        assert completion_row.prefix == completion_expected_prefixes[completion_key]
        completion_cost_path = completion_path(f"{completion_row.prefix}_dollar_cost.pkl")
        completion_cost_frame = pd.read_pickle(completion_cost_path)
        assert isinstance(completion_cost_frame, pd.DataFrame)
        assert completion_cost_frame.index.equals(tables[completion_row.split]["questions"].index)
        assert completion_cost_frame.columns.tolist() == model_names
        completion_cost_values = completion_cost_frame.to_numpy(dtype=float)
        assert np.isfinite(completion_cost_values).all() and (completion_cost_values > 0).all()
        pd.testing.assert_frame_equal(
            completion_cost_frame, completion_result["predictions"][completion_key],
        )
        completion_cost_predictions.append(completion_cost_path)

    completion_extension_paths = [completion_path(path) for path in completion_result["output_paths"]]
    completion_experiment_path = completion_path(completion_result["manifest_path"])
    completion_status_path = completion_path(completion_result["status_path"])
    completion_extension_manifest = json.loads(completion_experiment_path.read_text(encoding="utf-8"))
    completion_extension_status = json.loads(completion_status_path.read_text(encoding="utf-8"))
    assert completion_extension_status["status"] == "completed"
    assert completion_extension_status["workflow"] == completion_family
    assert completion_extension_status["mode"] == RUN_MODE
    assert completion_extension_status["run_id"] == run_stamp
    assert completion_extension_manifest["workflow"] == completion_family
    assert completion_extension_manifest["mode"] == RUN_MODE
    assert completion_extension_manifest["run_id"] == run_stamp
    assert completion_extension_manifest["model_names"] == model_names
    assert completion_extension_manifest["rows"] == completion_expected_rows
    assert completion_extension_manifest["test_used"] is False
    assert completion_extension_manifest["cost_only"] is True
    assert completion_extension_manifest["score_predictors_fitted"] is False
    assert set(completion_result["selected_alpha"]) == set(feature_labels)
    assert completion_extension_manifest["selected_alpha"] == completion_result["selected_alpha"]
    completion_saved_models = joblib.load(completion_path(f"{completion_family}_models.joblib"))
    assert completion_saved_models["model_names"] == model_names
    assert completion_saved_models["selected_alpha"] == completion_result["selected_alpha"]
    # Legacy Gamma files may predate the separate CV/final-fit alpha fields.
    # LogNormal's explicit E5 override must always have complete provenance.
    completion_alpha_fields = {"cv_selected_alpha", "fit_alpha", "final_fit_overrides"}
    if completion_family == "lognormal":
        assert all(completion_alpha_fields.issubset(record) for record in (
            completion_result, completion_extension_manifest, completion_saved_models))
    completion_cv_selected = completion_result.get("cv_selected_alpha", completion_result["selected_alpha"])
    completion_fit_alpha = completion_result.get("fit_alpha", completion_result["selected_alpha"])
    completion_fit_overrides = completion_result.get("final_fit_overrides", {})
    assert completion_cv_selected == completion_result["selected_alpha"]
    assert set(completion_fit_alpha) == set(feature_labels)
    assert completion_fit_alpha == {
        **completion_result["selected_alpha"],
        **{feature: override["alpha"] for feature, override in completion_fit_overrides.items()},
    }
    if completion_family == "lognormal":
        assert completion_fit_alpha["e5_388"] == 0.0
        assert completion_fit_overrides["e5_388"]["alpha"] == 0.0
        assert completion_fit_overrides["e5_388"]["reason"]
    for completion_record in (completion_extension_manifest, completion_saved_models):
        assert completion_record.get("cv_selected_alpha", completion_record["selected_alpha"]) == completion_cv_selected
        assert completion_record.get("fit_alpha", completion_record["selected_alpha"]) == completion_fit_alpha
        assert completion_record.get("final_fit_overrides", {}) == completion_fit_overrides
    completion_cv_summary = pd.read_csv(completion_path(f"{completion_family}_cv_summary.csv"))
    completion_final_fit = pd.read_csv(completion_path(f"{completion_family}_final_fit.csv"))
    for completion_feature in feature_labels:
        completion_cv_choices = completion_cv_summary.loc[completion_cv_summary["Feature"].eq(completion_feature)]
        completion_cv_best = completion_cv_choices.sort_values(
            ["Cost_WAPE", "alpha"], ascending=[True, False], kind="stable").iloc[0]
        assert float(completion_cv_best["alpha"]) == completion_result["selected_alpha"][completion_feature]
        completion_actual_alpha = float(completion_fit_alpha[completion_feature])
        completion_fit_rows = completion_final_fit.loc[completion_final_fit["Feature"].eq(completion_feature)]
        assert completion_fit_rows["Model"].tolist() == model_names
        assert completion_fit_rows["alpha"].eq(completion_actual_alpha).all()
        for completion_models in (completion_result["models"], completion_saved_models["models"]):
            assert len(completion_models[completion_feature]) == len(model_names)
            assert all(float(head["estimator"].alpha) == completion_actual_alpha
                       for head in completion_models[completion_feature])
    for completion_feature in feature_labels:
        assert (completion_feature, "calibration") in completion_result["predictions"]
        assert (completion_feature, "validation") in completion_result["predictions"]
    completion_artifacts.extend(completion_extension_paths)
    completion_artifacts.extend([completion_manifest_path, completion_experiment_path, completion_status_path])
    completion_extension_records[completion_family] = {
        "family": completion_label, "bundle_type": "cost_only",
        "selected_alpha": {key: float(value) for key, value in completion_result["selected_alpha"].items()},
        "cv_selected_alpha": {key: float(value) for key, value in completion_cv_selected.items()},
        "fit_alpha": {key: float(value) for key, value in completion_fit_alpha.items()},
        "final_fit_overrides": dict(completion_fit_overrides),
        "prediction_manifest": str(completion_manifest_path.relative_to(completion_root)).replace("\\", "/"),
        "experiment_manifest": str(completion_experiment_path.relative_to(completion_root)).replace("\\", "/"),
        "run_status": str(completion_status_path.relative_to(completion_root)).replace("\\", "/"),
        "prediction_files": 4,
        "rows": dict(completion_expected_rows),
        "model_names": list(model_names),
    }
    # Preserve provenance references without assuming a particular source-fingerprint schema.
    completion_source_records = {
        key: value for key, value in completion_extension_manifest.items()
        if any(word in key.lower() for word in ("source", "fingerprint", "sha256"))
    }
    if completion_source_records:
        completion_extension_records[completion_family]["source_provenance"] = completion_source_records

assert len(completion_cost_predictions) == len(set(completion_cost_predictions)) == 8
assert not set(completion_cost_predictions).intersection(completion_core_predictions)
completion_artifacts.extend(completion_cost_predictions)
completion_artifacts = sorted(set(completion_artifacts))

completion_global_manifest_path = run_dir / "experiment_manifest.json"
completion_global_manifest = json.loads(completion_global_manifest_path.read_text(encoding="utf-8"))
assert completion_global_manifest["mode"] == RUN_MODE
assert completion_global_manifest["run_id"] == run_stamp
assert completion_global_manifest["model_names"] == model_names
assert completion_global_manifest["rows"] == completion_expected_rows
completion_global_manifest.update({
    "additional_cost_predictors": completion_extension_records,
    "cost_comparison": {
        "split": "calibration", "averaging": "arithmetic mean of the 11 per-LLM metrics",
        "families": ["Ridge", "kNN", "XGBoost", "LogNormal", "Gamma"],
        "baseline": "per-LLM training-mean dollar cost",
        "per_llm_csv": cost_comparison_paths[0].name,
        "summary_csv": cost_comparison_paths[1].name,
    },
    "core_prediction_files": 208, "new_cost_prediction_files": 8, "prediction_files": 216,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
})
completion_global_manifest_path.write_text(
    json.dumps(completion_global_manifest, indent=2), encoding="utf-8",
)

# This is the only point that marks the complete merged workflow as completed.
run_status = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
assert run_status["workflow"] == "erdos_score_classification_router"
assert run_status["mode"] == RUN_MODE
assert run_status["run_id"] == run_stamp
run_status.update({
    "status": "completed", "stage": "all_experiments_completed",
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "refit_scope": "all_predictors_with_cost_extensions",
    "core_prediction_files": 208, "new_cost_prediction_files": 8, "prediction_files": 216,
    "additional_cost_predictors": ["lognormal", "gamma"],
    "validated_current_artifacts": len(completion_artifacts),
    "rows": dict(completion_expected_rows),
})
(run_dir / "run_status.json").write_text(json.dumps(run_status, indent=2), encoding="utf-8")
print("Completed mode:", RUN_MODE)
print("Active prediction files:", run_status["prediction_files"])
print("Experiment outputs:", run_dir)
if RUN_MODE == "smoke":
    print("Smoke validates execution. Use full mode for performance comparisons.")


Partial update 20261028T091427_279951Z_lognormal_e5_alpha0: only LogNormal E5 final/smearing fits rerun at alpha=0; CV and other predictors retained.
Active prediction files: 216
Experiment outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full
